**Settings.** Every path and tunable parameter in the notebook, grouped by the stage that uses it: file paths, filtering, the SCOWL dictionary, spaCy parsing, the data split, the three neural models, and evaluation.
Fixed word lists and lookup tables live next to the code that uses them.
After editing a value, re-run this cell and then the cells that use it.


In [1]:
from pathlib import Path

# ---------------------------------------------------------------- paths

TRAIN_CSV: Path = Path("train.csv")  # raw ASLG-PC12 export

# SCOWL download (contains final/, or a folder that does)
SCOWL_DIR: Path = Path("scowl-2020.12.07")

# Filtered data plus review/removed partitions
CACHE_PATH: Path = Path("cache/filtered.pkl")

# Readable TSV copies of the filtered, review, and removed partitions
FILTER_EXPORT_DIR: Path = Path("data/interim")

RESULTS_DIR: Path = Path("results/latest")  # Evaluation metrics and predictions

# Trained neural models, one file per system
MODEL_CACHE_DIR: Path = Path("cache/models")

# ---------------------------------------------------------------- general

RANDOM_SEED: int = 0  # Deterministic randomness for "choices"

# -------------------------------------------------------------- filtering

# Filter a random sample of N pairs; None for all.
FILTER_LIMIT: int | None = None

# Re-filter even if the cache matches these settings.
REBUILD_CACHE: bool = False

# Remove if fewer than this share of words are English.
NON_ENGLISH_MIN_RATIO: float = 0.5

# ------------------------------------------------------------------ SCOWL

# Largest SCOWL size level to load; larger lists add more non-English
# words, which would let foreign lines pass.
SCOWL_MAX_SIZE: int = 70

# ------------------------------------------------------- spaCy and models

# spaCy pipeline for parsing, and how many texts it parses per batch.
SPACY_MODEL: str = "en_core_web_sm"
SPACY_BATCH_SIZE: int = 256

# ----------------------------------------------------------------- splits

# Share of distinct English sentences held out from training.
# Rows with the same English always land in the same split, so test sentences
# are unseen.
VALIDATION_FRACTION: float = 0.1
TEST_FRACTION: float = 0.1

# ---------------------------------------------------------- neural models

# Torch device per model: "cpu", "cuda", "mps" (Apple GPU), or None for the
# fastest available (CUDA, then MPS, then CPU).
# On an M3 Max, MPS trains T5 2.7x faster than the CPU but runs the small LSTM
# 8x slower and the Transformer slightly slower.
# MPS needs torch 2.9.1 on macOS 14; newer torch versions crash there.
LSTM_DEVICE: str | None = "cpu"
TRANSFORMER_DEVICE: str | None = "cpu"
T5_DEVICE: str | None = None

# Sentences per batch when translating; larger batches decode faster (T5 on MPS:
# 5x faster at 128 than at 16).
# Training batches are set per model.
DECODE_BATCH_SIZE: int = 128
CLIP_NORM: float = 1.0  # gradient clipping, every neural model

# Train the neural models again even if a saved model matches.
RETRAIN_MODELS: bool = False

# Early stopping: training ends after this many epochs in a row without a
# new best validation loss, and the best epoch's weights are restored.
EARLY_STOPPING_PATIENCE: int = 3

# LSTM and Transformer only (T5 has its own subword tokenizer and limits).
# Source tokens seen fewer times in training become <unk>.
SEQ2SEQ_MIN_FREQ: int = 2

# Longer inputs are truncated, in words; bracketed parses run about 3x the
# sentence length.
SEQ2SEQ_MAX_SOURCE_LEN: int = 200
SEQ2SEQ_MAX_TARGET_LEN: int = 100  # also caps the length of generated gloss

# Train on batches of similar-length sentences (less padding): 1.3x faster for
# the LSTM and Transformer on the CPU.
# Off for T5, which is slower this way on the Apple GPU (it recompiles for every
# new batch shape).
SEQ2SEQ_BUCKET_BY_LENGTH: bool = True
T5_BUCKET_BY_LENGTH: bool = False

# ------------------------------------------------------------------- LSTM

# Model size and training schedule for the LSTM seq2seq.
LSTM_EMBED_DIM: int = 128  # word embeddings, source and target
LSTM_FACTOR_DIM: int = 16  # each extra flat-parse feature (POS, dep, head)
LSTM_HIDDEN_DIM: int = 256
LSTM_NUM_LAYERS: int = 1
LSTM_DROPOUT: float = 0.2
LSTM_EPOCHS: int = 50
LSTM_BATCH_SIZE: int = 32
LSTM_LEARNING_RATE: float = 1e-3

# ------------------------------------------------------------ Transformer

TRANSFORMER_D_MODEL: int = 256  # embedding and hidden size; divisible by heads

# Size of each extra flat-parse feature embedding (POS, dep, head).
TRANSFORMER_FACTOR_DIM: int = 16
TRANSFORMER_NUM_HEADS: int = 4
TRANSFORMER_ENCODER_LAYERS: int = 3
TRANSFORMER_DECODER_LAYERS: int = 3
TRANSFORMER_FF_DIM: int = 512  # feed-forward width inside each layer
TRANSFORMER_DROPOUT: float = 0.1
TRANSFORMER_EPOCHS: int = 50
TRANSFORMER_BATCH_SIZE: int = 64
TRANSFORMER_LEARNING_RATE: float = 5e-4

# --------------------------------------------------------------------- T5

# Pretrained checkpoint that T5Translator fine-tunes (downloaded once to the
# Hugging Face cache).
T5_MODEL: str = "google/flan-t5-small"
T5_EPOCHS: int = 3

# Batch 64 at 1e-3 trains 1.6-1.9x faster than 16 at 3e-4 on the Apple GPU,
# and reached a lower validation loss after one epoch.
T5_BATCH_SIZE: int = 64
T5_LEARNING_RATE: float = 1e-3

# Lengths in T5 subword tokens, not words; longer inputs are truncated.
T5_MAX_SOURCE_LEN: int = 512
T5_MAX_TARGET_LEN: int = 128

# ------------------------------------------------------------- evaluation

EVAL_LIMIT: int | None = None  # score a random sample of N pairs; None for all

# Pre-processing


## All imports and downloads

All required imports are placed here.
Note that PyTorch is pinned to 2.9.1 because newer versions crash on the Apple GPU (MPS) under macOS 14.


In [2]:
# Standard library.
import csv
import hashlib
import json
import math
import pickle
import re
import time
import unicodedata
from collections import Counter
from collections.abc import Callable, Iterable, Sequence
from dataclasses import asdict, dataclass
from itertools import dropwhile, takewhile
from pathlib import Path
from typing import Any, ClassVar, Self

# Data: DataFrames, plus BLEU and chrF scoring for evaluation.
import pandas as pd
import sacrebleu

# Plots of the training curves.
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# NLP: spaCy parsing (its type imports are just below), and its model
# downloader, imported from where it is defined so type checkers find it.
import spacy
from spacy.cli.download import download as download_spacy_model
from spacy.language import Language
from spacy.tokens import Token

# PyTorch for the LSTM, Transformer and T5 models.
# Pinned to 2.9.1: newer versions crash on the Apple GPU (MPS) under macOS 14.
import torch
from torch import Tensor, nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence

# Hugging Face transformers for the pretrained T5.
from transformers import (
    AutoTokenizer,
    BatchEncoding,
    PreTrainedTokenizerBase,
    T5ForConditionalGeneration,
)

/Users/skyeworster/Desktop/Linguistics/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1121 10:13:32.164000 70567 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Downloads the spaCy English model (`SPACY_MODEL`) only if it isn't installed yet, so re-running the notebook doesn't download it again.


In [3]:
# Download the spaCy English model only if it isn't installed yet.
if not spacy.util.is_package(SPACY_MODEL):
    print(f"{SPACY_MODEL} not detected; downloading.")
    download_spacy_model(SPACY_MODEL)

## Cleaning the input data

Reads the training data from `train.csv` into a DataFrame.
Byte-order marks and extra whitespace get dropped, and duplicates are removed.
Finally, it's sorted alphabetically.


In [4]:
# Read train.csv, an ordinary CSV with a "gloss,text" header.
# Every field is kept as text, even "NA" and "NULL"; quotes are removed.
raw_csv: pd.DataFrame = pd.read_csv(TRAIN_CSV, dtype=str, keep_default_na=False)

# Fields can start with a byte-order mark and end in \n; strip both.
stripped_csv: pd.DataFrame = pd.DataFrame(
    {
        column: raw_csv[column].str.replace("\ufeff", "").str.strip()
        for column in ("gloss", "text")
    }
)

# Rename text to english, drop repeated pairs, and sort by (gloss, english).
# The filtering cell below turns raw_data into `data`.
raw_data: pd.DataFrame = (
    stripped_csv.rename(columns={"text": "english"})
    .drop_duplicates()
    .sort_values(["gloss", "english"])
    .reset_index(drop=True)[["english", "gloss"]]
)
print(
    f"Removed {len(raw_csv) - len(raw_data)} duplicates. {len(raw_data)} pairs remain."
)

Removed 6619 duplicates. 81091 pairs remain.


## Filtering

I wanted to develop rules to automatically filter out undesirable pairs from the dataset.
The ultimate goal is to have two datasets: the usable `filtered.tsv` and the unusable `removed.tsv`.
The removal rules include:

- Pairs with non-Latin characters (the Greek alphabet is relatively common)
- Pairs with procedural jargon ("see minutes", "opening/closing of the sitting", "voting time")
- Gloss with non-terminal question marks, which are used to indicate something wasn't translated correctly (usually non-English names that use diacritics).
- Pairs with a high fraction of non-English words, as defined by the SCOWL dictionary.

Additionally, some rules simply flag pairs for review in `review.tsv`.
This is helpful for development, as it lets me see the effects of specific rules without having to trawl through everything in `removed.tsv`.

There are also checks that consider the entire dataset.
For example, `clean_dataframe()` finds all English lines that are identical except for the numbers within them.
It then randomly picks one pair to keep and drops the rest.

Some cleanup keeps pairs in the dataset with alterations, instead of flagging them.
For example, language codes ("es", "fr", "de") are removed from both sides of the pair.
A more serious correction is done to rectify the original machine translation's corruption of diacritics.
Each run of accented letters was rendered as `?` or junk, usually in place, but sometimes with the rest of the line written over the start of the gloss.
`clean_pairs()` restores the in-place cases, and the `garbled_accent` rule removes the pairs it can't restore.


### Filters

The non-pipeline stuff.


#### Repairing accents

The program that glossed ASLG-PC12 couldn't handle letters outside ASCII, so each run of accented letters became nothing, `?`, or up to two junk characters such as `à?`.
This happened in two ways:

- **In place**, the usual case: "rühle" became `R?HLE`. `clean_pairs()` (below) rewrites such tokens as the English word, uppercased (`RÜHLE`), before any other cleaning.
- **Overwritten**: the rest of the line, starting after the accented letters, was written over the start of the gloss, and `?` ended up last. For example, "our warmest thanks , harald rømer ." became `MER . THANKS , HARALD R ?`, losing `X-WE DESC-WARMEST`. The lost tokens can't be recovered, so the `garbled_accent` removal rule drops every pair whose gloss still lacks one of the English's accented words after the repair.

`accented_words()` lists the words both of these check. Words with no ASCII letter (Greek, a lone "à") are skipped, since a garbled form of them could be any token.


In [5]:
def accented_words(text: str) -> list[str]:
    """Word to repair must have both non-accented and accented characters."""
    return [
        word
        for word in dict.fromkeys(text.split())
        if any(ch.isalpha() and not ch.isascii() for ch in word)
        and any(ch.isalpha() and ch.isascii() for ch in word)
    ]

#### Cleanup

Cleanup that runs on every pair before the filters. `clean_pairs()`, in order:

1. restores accents garbled in place (see above);
2. turns repeated full stops (`..` or `. .`) into one;
3. strips spaces and punctuation from the start of each side, except opening brackets and quotes (`OPENERS`);
4. removes a language code that starts both sides ("de mr president" / `DE MR PRESIDENT`), including mangled gloss forms such as `C` for "cs";
5. removes a leading "subject" from both sides, but not "subject to".

Cleaning can make two pairs identical, so the Execution cell drops duplicates afterwards.
`normalize()` gives the form every filter rule reads: lowercase, single-spaced, no outer whitespace.


In [6]:
def normalize(text: str) -> str:
    r"""Ensures single-spaced, strips whitespace, makes lowercase.
    casefold() is lower() for all languages: "ß" -> "ss", for example."""
    return " ".join(text.casefold().split())


# Shouldn't be treated as punctuation for our purposes, despite Unicode.
NOT_PUNCTUATION: frozenset[str] = frozenset({"%", "&", "#", "@"})


def is_punctuation(ch: str) -> bool:
    """Is the character punctuation (with exceptions)"""
    return unicodedata.category(ch).startswith("P") and ch not in NOT_PUNCTUATION


# Opening and closing brackets and quotes.
OPENERS: frozenset[str] = frozenset("([{\"'«“‘")
CLOSERS: frozenset[str] = frozenset(")]}\"'»”’")


# Language codes placed at the beginning of lines ("de mr president , ...").
# The codes themselves are taken out, and the pairs continue being filtered.
# Italian "it" not included.
LANGUAGE_CODES: frozenset[str] = frozenset(
    {
        "bg", "cs", "da", "de", "el", "en", "es", "et", "fi", "fr", "ga", "hr",
        "hu", "lt", "lv", "mt", "nl", "pl", "pt", "ro", "sk", "sl", "sv",
    }  # (so that it doesn't expand)
)  # fmt: skip

# Mangled versions of language codes.
LANGUAGE_CODE_GLOSSES: dict[str, tuple[str, ...]] = {
    "fi": ("fus", "f"),
    "cs": ("c",),
    "es": ("e",),
}

# Every accepted "<code> <GLOSS>" pair of first words: the code, its first
# letter, or a mangled form, uppercased on the gloss side ("de DE", "cs C").
LANGUAGE_CODE_STARTS: frozenset[str] = frozenset(
    f"{code} {form.upper()}"
    for code in LANGUAGE_CODES
    for form in (code, code[0], *LANGUAGE_CODE_GLOSSES.get(code, ()))
)


def drop_first_word(df: pd.DataFrame, rows: pd.Series) -> pd.DataFrame:
    """Remove the first word of both sides of the selected pairs.

    Args:
        df (pd.DataFrame): Pairs with "english" and "gloss" columns.
        rows (pd.Series): One boolean per row of df; True where the first
            word goes.

    Returns:
        pd.DataFrame: Copy of df with those words removed (index kept).
    """
    stripped: pd.DataFrame = df.copy()
    for column in ("english", "gloss"):
        # Keep everything after the first word; a one-word side becomes "".
        stripped.loc[rows, column] = (
            df.loc[rows, column].str.split(n=1).str[1].fillna("")
        )
    return stripped


def clean_pairs(df: pd.DataFrame) -> pd.DataFrame:
    """Basic cleanup on pairs before filtering.
    Accents garbled in place by the gloss export are restored first.
    Repeated full stops become one.
    Leading punctuation, language codes, and "subject" lines are removed.

    Args:
        df (pd.DataFrame): Pairs with "english" and "gloss" columns.

    Returns:
        pd.DataFrame: A cleaned copy of df (index kept).
        Cleaning can make two pairs identical, so callers should drop duplicates
        afterwards.
    """

    # Two or more full stops, possibly spaced (".." or ". ."), become one.
    REPEATED_FULL_STOPS: re.Pattern[str] = re.compile(r"\.(?:\s*\.)+")

    cleaned: pd.DataFrame = df.copy()

    # Accents first, while the glosses are still as exported.
    glosses: list[str] = []
    for english, gloss in zip(df["english"], df["gloss"], strict=True):
        # One pattern per accented English word: every run of accented
        # letters became nothing, "?" or up to two junk characters.
        patterns: list[tuple[re.Pattern[str], str]] = [
            (
                re.compile(
                    ".{0,2}".join(map(re.escape, re.split(r"[^\x00-\x7f]+", word))),
                    re.IGNORECASE,
                ),
                word.upper(),
            )
            for word in accented_words(english)
        ]

        # No accented word, nothing to restore.
        if not patterns:
            glosses.append(gloss)
            continue
        tokens: list[str] = []
        for token in gloss.split():
            # A gloss prefix ("X-", "DESC-") is kept; only the rest is compared.
            prefix: str = m.group(0) if (m := re.match(r"(?:X|DESC)-", token)) else ""
            body: str = token[len(prefix) :]

            # A body with "?", non-ASCII junk or lowercase letters is garbled.
            # It becomes the first accented word it fits ("R?HLE" -> "RÜHLE").
            if "?" in body or not body.isascii() or not body.isupper():
                body = next(
                    (upper for pattern, upper in patterns if pattern.fullmatch(body)),
                    body,
                )
            tokens.append(prefix + body)
        glosses.append(" ".join(tokens))
    cleaned["gloss"] = glosses

    # Full stops, then leading spaces and punctuation, on both sides.
    # OPENERS are kept, since they start the text that follows.
    for column in ("english", "gloss"):
        cleaned[column] = [
            "".join(
                dropwhile(
                    lambda ch: ch.isspace()
                    or (is_punctuation(ch) and ch not in OPENERS),
                    text,
                )
            )
            for text in cleaned[column].str.replace(
                REPEATED_FULL_STOPS, ".", regex=True
            )
        ]

    # Lang codes: the first words of both sides ("de DE") form a known pair.
    # A missing first word gives NaN, which matches nothing.
    first_words: pd.Series = (
        cleaned["english"].str.split(n=1).str[0]
        + " "
        + cleaned["gloss"].str.split(n=1).str[0]
    )
    cleaned = drop_first_word(cleaned, first_words.isin(LANGUAGE_CODE_STARTS))

    # Nix "subject" as first word, but not "subject to".
    english_words: pd.Series = cleaned["english"].str.split(n=2)
    is_subject: pd.Series = (
        english_words.str[0].eq("subject")
        & english_words.str[1].ne("to")
        & cleaned["gloss"].str.split(n=1).str[0].eq("SUBJECT")
    )

    # Remove "subject", keep rest of line
    return drop_first_word(cleaned, is_subject)

#### Removal filters

Each rule takes the columns that `clean_dataframe()` prepares: english and gloss already normalized (lowercase, single-spaced, no outer whitespace), their word counts, and the share of English words.
The rule then returns one boolean per pair.

The rules are compiled into a dictionary, which is invoked in `clean_dataframe()` in the filtering pipeline.
This is used to record the reasons for removal in a separate column, which gets written to `removed.tsv`.

The rules in `REMOVE_RULES`:

| Rule                        | Removes pairs where                                                                                                    |
| --------------------------- | ---------------------------------------------------------------------------------------------------------------------- |
| `empty_side`                | either side has no word, only punctuation or nothing                                                                   |
| `name_report`               | the whole English line is `<name> report` or `report <name>`                                                           |
| `single_word`               | either side is a single word                                                                                           |
| `non_latin`                 | either side has Greek, Cyrillic or other non-Latin letters (accented Latin letters are fine)                           |
| `boilerplate`               | the English contains a meeting-minutes phrase from `EXCLUDED_PHRASES`                                                  |
| `spoken_url`                | either side spells out a web address ("europarl dot eu")                                                               |
| `gloss_inner_question_mark` | the gloss has a `?` followed by a letter, a digit or another `?`: almost always a character the export couldn't encode |
| `garbled_accent`            | the English has an accented word the gloss lacks after `clean_pairs()` restores accents                                |
| `non_english`               | fewer than `NON_ENGLISH_MIN_RATIO` of the English words are in the SCOWL dictionary                                    |


In [7]:
# "<name> report" with a one- or two-word name
# or "report <name>" with up to four words after "report"
REPORT_LINE: re.Pattern[str] = re.compile(
    r"^(?:\S+(?: \S+)? report|report \S+(?: \S+){0,3})(?: \.)?$"
)

# A "?" followed later by a letter or digit, or by another "?".
# A plain "? !" at the end still counts as terminal punctuation.
# In ASLG-PC12 gloss this is almost always a letter the export couldn't encode.
INNER_QUESTION_MARK: re.Pattern[str] = re.compile(r"\?(?=.*[^\W_]|\s*\?)")

# Top-level domains caught when spoken aloud ("dot com", "dot eu").
SPOKEN_TLDS: list[str] = ["com", "eu", "org", "net", "int", "gov", "edu", "info", "uk"]

# "dot" followed by a top-level domain, as in "europarl dot eu".
SPOKEN_URL: re.Pattern[str] = re.compile(rf"\bdot (?:{'|'.join(SPOKEN_TLDS)})\b")


def has_non_latin_letter(text: str) -> bool:
    """True if any letter is outside the Latin script (diacritics are fine)."""

    # Only letters are checked, so symbols like "€" or "°" don't count.
    return not text.isascii() and any(
        ch.isalpha() and not unicodedata.name(ch, "").startswith("LATIN") for ch in text
    )


# Boilerplate from meeting minutes, standard parliamentary headings.
# Regex match in English text removes pair.
EXCLUDED_PHRASES: list[str] = [
    r"see minutes",
    r"(?:opening|closing) of the sitting",
    r"voting time",
    r"approval of the minutes",
    r"resumption of the session",
    r"adjournment of the session",
    r"agenda for the next sitting",
]

# Any excluded phrase as whole words.
BOILERPLATE_PATTERN: re.Pattern[str] = re.compile(
    r"\b(?:" + "|".join(f"(?:{p})" for p in EXCLUDED_PHRASES) + r")\b"
)

# Each rule is a function called on a DataFrame.
# The output is a boolean series (did the rule catch this pair?)
REMOVE_RULES: dict[str, Callable[[pd.DataFrame], pd.Series]] = {
    # Either side has no word at all, only punctuation or nothing.
    "empty_side": lambda t: (t["english_words"] == 0) | (t["gloss_words"] == 0),
    # The whole English line is "<name> report" or "report <name>".
    "name_report": lambda t: t["english"].str.match(REPORT_LINE),
    # Either side is a single word.
    "single_word": lambda t: (t["english_words"] == 1) | (t["gloss_words"] == 1),
    # Greek, Cyrillic and other non-Latin letters on either side.
    "non_latin": lambda t: t["english"].map(has_non_latin_letter)
    | t["gloss"].map(has_non_latin_letter),
    # Meeting-minutes phrases from EXCLUDED_PHRASES.
    "boilerplate": lambda t: t["english"].str.contains(BOILERPLATE_PATTERN),
    # Spoken links ("dot eu") on either side.
    "spoken_url": lambda t: t["english"].str.contains(SPOKEN_URL)
    | t["gloss"].str.contains(SPOKEN_URL),
    # A "?" inside the gloss, usually a garbled accented letter.
    "gloss_inner_question_mark": lambda t: t["gloss"].str.contains(INNER_QUESTION_MARK),
    # An accented English word the gloss lacks even after clean_pairs()
    # restored accents: the export overwrote part of the line or garbled the
    # word for good.
    "garbled_accent": lambda t: pd.Series(
        [
            any(word not in gloss for word in accented_words(english))
            for english, gloss in zip(t["english"], t["gloss"], strict=True)
        ],
        index=t.index,
    ),
    # Mostly non-English words (needs a dictionary; NaN never fires).
    "non_english": lambda t: t["english_ratio"] < NON_ENGLISH_MIN_RATIO,
}

#### Review filters

Like the removal filters above, but sends the output to a separate file for review.
Immensely useful during development, as it helps to develop stronger and more accurate filters.

The rules in `REVIEW_RULES`:

| Rule                            | Sends pairs to review where                                                                                |
| ------------------------------- | ---------------------------------------------------------------------------------------------------------- |
| `identical`                     | both sides are the same text of more than one word                                                         |
| `consecutive_punctuation`       | the sides have different runs of punctuation (`. .`, `? !`); the same run on both sides is fine            |
| `terminal_punctuation_mismatch` | the sentence-final punctuation differs, or only one side has any                                           |
| `unbalanced_brackets`           | either side has unmatched brackets                                                                         |
| `language_code`                 | only the English starts with a language code (codes on both sides were already removed by `clean_pairs()`) |

`punctuation_runs()` doesn't count a closing bracket followed by punctuation (`x ) .`) or punctuation followed by an opening bracket (`, ( x`) as a run, since both are normal.


In [8]:
def punctuation_runs(text: str) -> list[str]:
    """Every run of two or more punctuation marks, spaces ignored (". ?" ->
    ".?").
    A closer followed by punctuation ("x ) .") or punctuation followed by an
    opener (", ( x") is normal, so it ends a run instead."""
    runs: list[str] = []
    current: str = ""
    for ch in text:
        if ch.isspace():
            continue

        # Extend the run unless this is a normal closer/opener boundary.
        if is_punctuation(ch) and not (
            current and (current[-1] in CLOSERS or ch in OPENERS)
        ):
            current += ch
            continue

        # The run ends here; a punctuation mark that broke it starts the next.
        if len(current) > 1:
            runs.append(current)
        current = ch if is_punctuation(ch) else ""

    # A run can also end with the text.
    if len(current) > 1:
        runs.append(current)
    return runs


def terminal_punctuation(text: str) -> str:
    """The last character if it is punctuation, else "".
    On normalized text this treats ".." and "agreement." the same as " ."."""
    return text[-1] if text and is_punctuation(text[-1]) else ""


# Bracket types whose openers and closers must balance.
BRACKET_PAIRS: tuple[tuple[str, str], ...] = (("(", ")"), ("[", "]"), ("{", "}"))


def has_unbalanced_brackets(text: str) -> bool:
    """True if any BRACKET_PAIRS type has more openers than closers or vice
    versa."""
    return any(
        text.count(opener) != text.count(closer) for opener, closer in BRACKET_PAIRS
    )


REVIEW_RULES: dict[str, Callable[[pd.DataFrame], pd.Series]] = {
    # The same text on both sides, but more than one word.
    "identical": lambda t: (t["english"] == t["gloss"]) & (t["english_words"] > 1),
    # Runs like ". ." or "? !" on either side.
    # Only when the two sides differ; the same run on both sides is fine.
    "consecutive_punctuation": lambda t: pd.Series(
        [
            punctuation_runs(english) != punctuation_runs(gloss)
            for english, gloss in zip(t["english"], t["gloss"], strict=True)
        ],
        index=t.index,
    ),
    # Sentence-final punctuation differs ("?" vs "." or one side has none).
    "terminal_punctuation_mismatch": lambda t: t["english"].map(terminal_punctuation)
    != t["gloss"].map(terminal_punctuation),
    # Unmatched brackets on either side.
    "unbalanced_brackets": lambda t: t["english"].map(has_unbalanced_brackets)
    | t["gloss"].map(has_unbalanced_brackets),
    # A language code on the English side only; codes on both sides were
    # already removed by clean_pairs().
    "language_code": lambda t: t["english"].str.split(n=1).str[0].isin(LANGUAGE_CODES),
}

#### Rule execution

`clean_dataframe()` runs every rule on every pair and splits the pairs into the three partitions:

1. Both sides are normalized once, and their word counts and the share of English words (from SCOWL) are added; every rule reads these columns.
2. All removal and review rules run. Removal takes priority over review.
3. Number variants are looked for among the pairs that weren't removed, so removed lines can't make a kept line look like a variant. These are English sentences that differ only in their numbers ("written statements rule 142" / "... rule 120"). One pair per group is kept (chosen at random with `RANDOM_SEED`), and the rest are removed as `number_variant`.
4. Pairs that would go to review but start with a number are removed instead (`starts_with_number`). I checked all of these by hand to ensure they're garbage.

Each flagged pair's `reasons` column lists every rule that fired on it.
It returns four DataFrames: the kept, review and removed pairs, and how often each rule fired.


In [9]:
def clean_dataframe(
    df: pd.DataFrame,
    english_words: frozenset[str],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Run every rule on every pair and partition the DataFrame.

    Args:
        df (pd.DataFrame): Pairs with "english" and "gloss" columns.
        english_words (frozenset[str]): Lowercase English words for the
            non_english rule.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]: The kept,
        review and removed rows (original index kept; review and removed
        have a reasons column), and how often each reason fired.
    """

    # Regex to find words (without punctuation)
    WORD_TOKEN: str = r"\S*[^\W_]\S*"

    # The normalized text every rule reads, plus word counts.
    columns: pd.DataFrame = pd.DataFrame(
        {"english": df["english"].map(normalize), "gloss": df["gloss"].map(normalize)},
        index=df.index,
    )

    # Record the sentence lengths.
    columns["english_words"] = columns["english"].str.count(WORD_TOKEN)
    columns["gloss_words"] = columns["gloss"].str.count(WORD_TOKEN)

    # Column for English word ratios.
    columns["english_ratio"] = math.nan

    # Hyphens and apostrophes split words; skip one-letter pieces.
    words: pd.Series = columns["english"].str.findall(r"[^\W\d_]{2,}")

    # Count each row's dictionary words (one row per word, regrouped).
    known: pd.Series = words.explode().isin(english_words).groupby(level=0).sum()

    # Known words over all words; 0 / 0 (no words) gives NaN.
    columns["english_ratio"] = known / words.str.len()

    # One boolean column per reason, in rule order.
    flags: pd.DataFrame = pd.DataFrame(
        {
            reason: rule(columns)
            for reason, rule in (REMOVE_RULES | REVIEW_RULES).items()
        },
        index=df.index,
    ).astype(bool)
    removed: pd.Series = flags[list(REMOVE_RULES)].any(axis=1)

    # Number variants: survivors that differ only in their numbers.
    # A number can be joined by . , : / or - ("1.5", "2009-2014").
    NUMBER: re.Pattern[str] = re.compile(r"\d+(?:[.,:/-]\d+)*")
    survivors: pd.Series = columns.loc[~removed, "english"]

    # Same text with every number replaced, e.g. "rule <num>".
    template: pd.Series = survivors.str.replace(NUMBER, "<num>", regex=True)
    has_number: pd.Series = survivors.str.contains(NUMBER)

    # Rows whose template is shared by at least two distinct texts (exact
    # duplicates alone don't count).
    distinct: pd.Series = (
        survivors[has_number].groupby(template[has_number]).transform("nunique")
    )
    variants: pd.Index = distinct.index[distinct > 1]

    # One seeded random row per template is kept; the others are removed.
    keep: pd.Index = (
        template[variants]
        .to_frame("template")
        .groupby("template")
        .sample(n=1, random_state=RANDOM_SEED)
        .index
    )
    flags["number_variant"] = df.index.isin(variants) & ~df.index.isin(keep)
    removed |= flags["number_variant"]

    # Pairs that would go to review but start with a number are removed.
    flags["starts_with_number"] = (
        flags.any(axis=1) & ~removed & columns["english"].str.match(r"\d")
    )
    removed |= flags["starts_with_number"]

    # Each row's reasons as one comma-separated string.
    with_reasons: pd.DataFrame = df.assign(
        reasons=[",".join(flags.columns[row]) for row in flags.to_numpy()]
    )
    flagged: pd.Series = flags.any(axis=1)

    # How often each reason fired, most frequent first (ties in rule order).
    counts: pd.Series = flags.sum()
    counts = counts[counts > 0].sort_values(ascending=False, kind="stable")

    # kept, review, removed, and how often each reason fired.
    return (
        df.loc[~flagged].copy(),
        with_reasons.loc[flagged & ~removed],
        with_reasons.loc[removed],
        counts.rename_axis("reason").reset_index(name="count"),
    )

### Pipeline

`print_summary()` prints the size of each partition and how often each rule fired, from the four DataFrames `clean_dataframe()` returns.
The pipeline itself runs in the Execution cell below.


In [10]:
def print_summary(
    result: tuple[pd.DataFrame, ...],
) -> None:
    """Print clean_dataframe()'s partition sizes and how often each reason
    fired."""
    kept, review, removed, counts = result
    print(f"kept {len(kept)}, review {len(review)}, removed {len(removed)}")

    # One line per reason, sorted by frequency.
    for reason, count in zip(counts["reason"], counts["count"], strict=True):
        print(f"  {reason:<32}{count:>8}")

### Test filters

A demonstration of the filters on hand-made pairs that each trip at least one rule, using a tiny stand-in dictionary instead of SCOWL.
It runs `clean_pairs()` and then `clean_dataframe()`, as the Execution cell does, then prints each flagged pair with the rules that fired, followed by the pairs that were kept.


In [11]:
# Filter a hand-made set of pairs against a small dictionary, to show what
# each rule catches.
demo_words: frozenset[str] = frozenset(
    {
        "the", "commission", "must", "act", "now", "democracy", "report",
        "written", "statements", "rule", "subject", "mr", "president", "we",
        "agree", "vote", "thank", "with", "mrs",
    }
)  # fmt: skip

demo_pairs: pd.DataFrame = pd.DataFrame(
    [
        ("the commission must act now .", "COMMISSION MUST ACT NOW ."),
        # Shared language code: stripped from both sides, then kept.
        ("de mr president , we agree .", "DE MR PRESIDENT , X-WE AGREE ."),
        ("lange report", "LANGE REPORT"),
        ("report bogdan golik", "REPORT BOGDAN GOLIK"),
        ("vote", "VOTE"),
        (
            "membership of parliament see minutes",
            "MEMBERSHIP PARLIAMENT SEE MINUTE",
        ),
        (
            "visit www.europarl.europa.eu for details",
            "VISIT WWW.EUROPARL.EUROPA.EU DETAIL",
        ),
        ("visit europarl dot eu", "VISIT EUROPARL DOT EU"),
        ("Ελλάδα is a country", "ΕΛΛΆΔΑ BE COUNTRY"),
        ("café crème is fine", "CAFÉ CRÈME FINE"),
        ("do we agree ? !", "X-WE AGREE ?"),
        ("subject democracy report", "SUBJECT DEMOCRACY REPORT"),
        # Accented letters garbled to "?" in the gloss: removed.
        ("thank you , mrs záborská .", ". ? , MR Z ?"),
        # Mangled code ("cs" -> "C"), doubled full stop: cleaned, then kept.
        ("cs we agree ..", "C X-WE AGREE ."),
        # The same "?!" on both sides is not flagged.
        ("we agree ? !", "X-WE AGREE ? !"),
        # Review reason plus a leading number: removed.
        ("3 . subject democracy report ?", "3 . DEMOCRACY REPORT ."),
        # Language code on the English side only: sent to review.
        ("cs mr president , we agree .", "MR PRESIDENT , X-WE AGREE ."),
        ("written statements rule 142", "WRITTEN STATEMENT RULE 142"),
        ("written statements rule 120", "WRITTEN STATEMENT RULE 120"),
        ("die kommission muss handeln", "DIE KOMMISSION MUSS HANDELN"),
        # Accent garbled in place: restored, then kept.
        ("we agree with mrs rühle .", "X-WE AGREE WITH MR R?HLE ."),
        # Gloss overwritten from the accent on ("X-WE MUST" lost): removed.
        ("we must thank mrs malmström .", "M . THANK MR MALMSTR ?"),
    ],
    columns=["english", "gloss"],
)

# The same steps the filtering cell runs: clean_pairs(), then filter.
demo_result: tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame] = (
    clean_dataframe(clean_pairs(demo_pairs), demo_words)
)
print_summary(demo_result)
demo_kept, demo_review, demo_removed, _ = demo_result

# Each flagged pair with the reasons that fired, then the kept pairs.
filter_demo_df: pd.DataFrame = pd.concat([demo_removed, demo_review])
for reasons, english in zip(
    filter_demo_df["reasons"], filter_demo_df["english"], strict=True
):
    print(f"{reasons:<55} {english}")
for english in demo_kept["english"]:
    print(f"{'(kept)':<55} {english}")

kept 6, review 2, removed 14
  non_english                            7
  identical                              5
  terminal_punctuation_mismatch          4
  name_report                            3
  garbled_accent                         2
  single_word                            1
  non_latin                              1
  boilerplate                            1
  spoken_url                             1
  consecutive_punctuation                1
  language_code                          1
  number_variant                         1
  starts_with_number                     1
name_report,identical                                   lange report
name_report,non_english,identical                       report bogdan golik
single_word                                             vote
boilerplate,non_english                                 membership of parliament see minutes
non_english                                             visit www.europarl.europa.eu for details
spoken_url,non_e

### Execution

The entire filtering step.
It starts with `clean_pairs()`, then drops the duplicates, then runs `clean_dataframe()` with the SCOWL dictionary.
`FILTER_LIMIT` filters a random sample instead of every pair.

The dictionary is the SCOWL word lists in one lowercase set, which the `non_english` rule checks English words against.
It includes the spellings in `SCOWL_SPELLINGS` and the lists in `SCOWL_SUBCATEGORIES` (ordinary words, capitalized words, proper names, abbreviations), up to size level `SCOWL_MAX_SIZE`.
Larger size levels add rarer words, including foreign ones that would let non-English lines pass.

The result is cached in `CACHE_PATH` and reused as long as `train.csv` and every setting that affects filtering are unchanged.
`FILTER_RULES_VERSION` is **manually** bumped whenever a rule changes.
The cell also writes `filtered.tsv`, `review.tsv`, `removed.tsv` and `summary.json` to `FILTER_EXPORT_DIR`.

Every later cell uses the kept pairs as `data`; `review_df` and `removed_df` hold the rest, and `reason_counts` how often each rule fired.
Re-running this cell is safe, since `raw_data` itself is never changed.


In [12]:
# Filter the loaded pairs; every later cell uses the kept rows as `data`.
# Re-running this cell is safe: raw_data itself is never changed.
# The cached result is reused only when train.csv and every setting that
# affects filtering are unchanged.
# filtered.tsv, review.tsv, removed.tsv and summary.json are written to
# FILTER_EXPORT_DIR either way.

# Part of the cache fingerprint.
# MUST BE MANUALLY INCREMENTED.
FILTER_RULES_VERSION: int = 9

# SCOWL's size levels, smallest to largest (see SCOWL_MAX_SIZE).
SCOWL_SIZES: tuple[int, ...] = (10, 20, 35, 40, 50, 55, 60, 70, 80, 95)

# Spelling variants to include
SCOWL_SPELLINGS: tuple[str, ...] = (
    "english","american","british","variant_1", "british_variant_1",
) # fmt: skip

# Include sub-dicts for proper names, places, etc.
SCOWL_SUBCATEGORIES: tuple[str, ...] = (
    "words","upper","proper-names","abbreviations",
) #fmt:skip

# Fingerprint train.csv (path, size, modified time) plus the settings
# that change the output; any difference forces a rebuild.
stat = TRAIN_CSV.stat()
fingerprint: str = hashlib.sha256(
    json.dumps(
        {
            "source": [str(TRAIN_CSV.resolve()), stat.st_size, stat.st_mtime_ns],
            "settings": [
                FILTER_RULES_VERSION,
                RANDOM_SEED,
                FILTER_LIMIT,
                NON_ENGLISH_MIN_RATIO,
                EXCLUDED_PHRASES,
                SPOKEN_TLDS,
                sorted(LANGUAGE_CODE_STARTS),
                sorted(NOT_PUNCTUATION),
                sorted(OPENERS),
                sorted(CLOSERS),
                BRACKET_PAIRS,
                SCOWL_MAX_SIZE,
                SCOWL_SPELLINGS,
                SCOWL_SUBCATEGORIES,
            ],
        }
    ).encode("utf-8")
).hexdigest()

# The cache keys, in the order clean_dataframe() returns the partitions.
partitions: tuple[str, ...] = ("kept", "review", "removed", "reason_counts")

# Set from the cache if it is reused, otherwise built below: the kept,
# review and removed pairs, and the reason counts.
filter_result: tuple[pd.DataFrame, ...] | None = None

# Check old cache (if it exists)
if CACHE_PATH.is_file() and not REBUILD_CACHE:
    cached: dict[str, Any] = pickle.loads(CACHE_PATH.read_bytes())

    # If cache isn't stale, reuse it.
    if cached.get("fingerprint") == fingerprint:
        filter_result = (
            cached["kept"],
            cached["review"],
            cached["removed"],
            cached["reason_counts"],
        )
        print(f"Loaded filtered data from {CACHE_PATH}")

# If the cache is stale, build from scratch.
if filter_result is None:
    stripped: pd.DataFrame = clean_pairs(raw_data)  # Pre-processing.
    source: pd.DataFrame = stripped.drop_duplicates()  # Drop dupes.

    # Count the pairs that cleaning changed on either side.
    changed: int = (
        (stripped[["english", "gloss"]] != raw_data[["english", "gloss"]])
        .any(axis=1)
        .sum()
    )
    print(f"Cleaned {changed} pairs; {len(stripped) - len(source)} became duplicates.")

    # Pull a random selection of rows (or use them all).
    if FILTER_LIMIT is not None and FILTER_LIMIT < len(source):
        source = source.sample(n=FILTER_LIMIT, random_state=RANDOM_SEED).sort_index()

    # SCOWL word lists up to SCOWL_MAX_SIZE, for the non_english rule.
    if SCOWL_MAX_SIZE not in SCOWL_SIZES:
        raise ValueError(
            f"SCOWL_MAX_SIZE must be one of {SCOWL_SIZES}, got {SCOWL_MAX_SIZE}"
        )

    # The word files sit in final/, whichever folder SCOWL_DIR names.
    scowl_final: Path = SCOWL_DIR if SCOWL_DIR.name == "final" else SCOWL_DIR / "final"

    # Files are named <spelling>-<subcategory>.<size>.
    # Not every spelling has every subcategory at every size, so missing
    # files are skipped.
    scowl_paths: list[Path] = [
        scowl_final / f"{spelling}-{subcategory}.{size}"
        for spelling in SCOWL_SPELLINGS
        for subcategory in SCOWL_SUBCATEGORIES
        for size in SCOWL_SIZES
        if size <= SCOWL_MAX_SIZE
    ]
    scowl_paths = [path for path in scowl_paths if path.is_file()]
    if not scowl_paths:
        raise FileNotFoundError(f"No SCOWL word files matched in {scowl_final}")

    # One word per line, encoded in ISO-8859-1, not UTF-8; skip blank lines.
    scowl_words: frozenset[str] = frozenset(
        word
        for path in scowl_paths
        for line in path.read_text(encoding="iso-8859-1").splitlines()
        if (word := line.strip().casefold())
    )
    filter_result = clean_dataframe(source, scowl_words)  # Filter w/ SCOWL.

    # Make the cache directory, if it doesn't exist.
    CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)

    # Cache all partitions.
    CACHE_PATH.write_bytes(
        pickle.dumps(
            {"fingerprint": fingerprint}
            | dict(zip(partitions, filter_result, strict=True))
        )
    )
    print(f"Filtered {len(source)} pairs and cached the result to {CACHE_PATH}")
data, review_df, removed_df, reason_counts = filter_result

# Readable copies of every partition, keyed by the original row id.
FILTER_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
for name, frame in (("filtered", data), ("review", review_df), ("removed", removed_df)):
    frame.to_csv(FILTER_EXPORT_DIR / f"{name}.tsv", sep="\t", index_label="row")

# Partition sizes and reason counts as JSON.
summary: dict[str, object] = {
    "kept": len(data),
    "review": len(review_df),
    "removed": len(removed_df),
    # tolist() gives plain Python ints, which json can write.
    "reason_counts": dict(
        zip(
            reason_counts["reason"].tolist(),
            reason_counts["count"].tolist(),
            strict=True,
        )
    ),
}

# Write to export path.
(FILTER_EXPORT_DIR / "summary.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8"
)
print(
    f"Wrote filtered.tsv, review.tsv, removed.tsv and summary.json to {FILTER_EXPORT_DIR}"
)
print_summary(filter_result)

Loaded filtered data from cache/filtered.pkl
Wrote filtered.tsv, review.tsv, removed.tsv and summary.json to data/interim
kept 79159, review 393, removed 1503
  garbled_accent                       548
  identical                            525
  number_variant                       331
  terminal_punctuation_mismatch        284
  boilerplate                          245
  name_report                          187
  single_word                          130
  non_english                           86
  gloss_inner_question_mark             79
  consecutive_punctuation               75
  language_code                         33
  starts_with_number                    25
  non_latin                             19
  empty_side                             4


# Parse tree

Turns spaCy dependency parses into strings a seq2seq model can read, in two formats:

- **bracketed**, a nested, tree-shaped string: `(ROOT eat/VERB (nsubj i/PRON) (dobj apple/NOUN (det the/DET)))`
- **flat**, one `lemma|POS|dep|head` chunk per token, in sentence order: `i|PRON|nsubj|1 eat|VERB|ROOT|1 the|DET|det|3 apple|NOUN|dobj|1`

`parse_token()` gives a token's four fields (lowercase lemma, POS tag, dependency label and head index) as a tuple of strings.
`bracket_token()` builds the bracketed string recursively, and also returns the subtree's tokens in tree order: each head before its dependents.
`keep_token()` leaves punctuation out of both formats.


In [13]:
def parse_token(token: Token) -> tuple[str, ...]:
    """The four parse fields of a spaCy token, all as strings.

    Args:
        token (Token): A token from a parsed spaCy Doc.

    Returns:
        tuple[str, ...]: (lemma, POS, dep, head): the lowercase lemma, POS
        tag, dependency label, and the head's index in the sentence.
        Dropped punctuation is not renumbered, so indices match spaCy's.

    Example:
        >>> doc = spacy.load(SPACY_MODEL)("I eat the apple.")
        >>> parse_token(doc[0])
        ('i', 'PRON', 'nsubj', '1')
    """
    return (
        token.lemma_.lower(),
        token.pos_,
        token.dep_,
        str(token.head.i - token.sent.start),
    )


def keep_token(token: Token) -> bool:
    """Does the token go into the parses?
    Punctuation is left out."""
    return token.dep_ != "punct"


def bracket_token(token: Token) -> tuple[str, list[tuple[str, ...]]]:
    """Recursively render a token and its dependents as nested brackets.
    spaCy yields children in sentence order (left dependents, then right),
    so the bracketed string preserves the original English word order.

    Args:
        token (Token): The subtree's head, usually a sentence root.

    Returns:
        tuple[str, list[tuple[str, ...]]]: "(dep lemma/POS child ...)", with
        lowercase lemmas, and the subtree's parse_token() tuples in the same
        order: each head before its dependents, depth first.

    Example:
        >>> doc = spacy.load(SPACY_MODEL)("I eat the apple.")
        >>> text, tokens = bracket_token(doc[1])
        >>> text
        '(ROOT eat/VERB (nsubj i/PRON) (dobj apple/NOUN (det the/DET)))'
        >>> [lemma for lemma, *_ in tokens]
        ['eat', 'i', 'apple', 'the']
    """
    # Make this token the head.
    head: tuple[str, ...] = parse_token(token)
    lemma, pos, dep, _ = head

    # Render each kept child's own subtree, in sentence order.
    children: list[tuple[str, list[tuple[str, ...]]]] = [
        bracket_token(child) for child in token.children if keep_token(child)
    ]

    # String representation.
    text: str = " ".join([f"({dep} {lemma}/{pos}", *(child for child, _ in children)])
    tree: list[tuple[str, ...]] = [
        head,
        *(t for _, tokens in children for t in tokens),
    ]
    return text + ")", tree

### Sentence parser

`parse_sentences()` parses English with spaCy (`parser_nlp`) and returns both formats, as strings and as token tuples, one row per sentence.
Each parse is kept in `parse_cache`, keyed by the sentence, so each sentence is parsed only once, however many translators read it.
A line with several sentences has each one serialized and joined with `|||`.
The cell ends by printing the parses of the first 10 sentences.


In [14]:
# The spaCy pipeline for parse trees.
# NER is not needed for a dependency parse, so skip it for speed.
parser_nlp: Language = spacy.load(SPACY_MODEL, disable=["ner"])


# Parses seen so far, looked up by the English sentence.
# Each holds the four columns parse_sentences() returns besides english.
parse_cache: dict[str, dict[str, Any]] = {}


def parse_sentences(sentences: Sequence[str]) -> pd.DataFrame:
    """Parse English sentences, reusing the parses from earlier calls.

    Only sentences not already in parse_cache are parsed.
    A text with several sentences has each one serialized and joined with
    SENTENCE_SEPARATOR (" ||| "), and all their tokens in one list.

    Args:
        sentences (Sequence[str]): English sentences to parse.

    Returns:
        pd.DataFrame: One row per sentence, in input order, duplicates
        included: english, bracketed, flat, and each format's
        parse_token() tuples (bracketed_tokens, flat_tokens).

    Example:
        >>> parsed = parse_sentences(["I eat the apple.", "I eat the apple."])
        >>> parsed["bracketed"][0]
        '(ROOT eat/VERB (nsubj i/PRON) (dobj apple/NOUN (det the/DET)))'
        >>> parsed["flat"][0]
        'i|PRON|nsubj|1 eat|VERB|ROOT|1 the|DET|det|3 apple|NOUN|dobj|1'
        >>> len(parsed)
        2
        >>> print(parse_sentences(["I run. You walk."])["flat"][0])
        i|PRON|nsubj|1 run|VERB|ROOT|1 ||| you|PRON|nsubj|1 walk|VERB|ROOT|1
    """

    # Joins the parses of a text with several sentences.
    SENTENCE_SEPARATOR: str = " ||| "

    # The distinct sentences not parsed yet, in first-seen order.
    missing: list[str] = list(
        dict.fromkeys(s for s in sentences if s not in parse_cache)
    )

    # Parse them in batches; pipe() yields the docs in input order.
    docs = parser_nlp.pipe(missing, batch_size=SPACY_BATCH_SIZE)
    for sentence, doc in zip(missing, docs, strict=True):
        # Per sentence: its bracketed string with its tokens in tree order,
        # and its kept tokens in sentence order.
        trees: list[tuple[str, list[tuple[str, ...]]]] = [
            bracket_token(sent.root) for sent in doc.sents
        ]
        flat: list[list[tuple[str, ...]]] = [
            [parse_token(t) for t in sent if keep_token(t)] for sent in doc.sents
        ]

        # A text with several sentences becomes one separated string per
        # format, and one token list per format.
        parse_cache[sentence] = {
            "bracketed": SENTENCE_SEPARATOR.join(text for text, _ in trees),
            "flat": SENTENCE_SEPARATOR.join(
                " ".join("|".join(t) for t in tokens) for tokens in flat
            ),
            "bracketed_tokens": [t for _, tokens in trees for t in tokens],
            "flat_tokens": [t for tokens in flat for t in tokens],
        }

    # Look up every requested sentence, duplicates included, in order.
    # Naming the columns keeps them when sentences is empty.
    return pd.DataFrame(
        [{"english": s, **parse_cache[s]} for s in sentences],
        columns=["english", "bracketed", "flat", "bracketed_tokens", "flat_tokens"],
    )


# Preview both parse formats for the first 10 filtered sentences.
parse_samples_df: pd.DataFrame = parse_sentences(data["english"].head(10).tolist())
for text, bracketed, flat in zip(
    parse_samples_df["english"],
    parse_samples_df["bracketed"],
    parse_samples_df["flat"],
    strict=True,
):
    print(f"TEXT:      {text}")
    print(f"BRACKETED: {bracketed}")
    print(f"FLAT:      {flat}")
    print()

TEXT:      mr deß , I will explain .
BRACKETED: (ROOT explain/VERB (dep deß/PROPN (compound mr/PROPN)) (nsubj i/PRON) (aux will/AUX))
FLAT:      mr|PROPN|compound|1 deß|PROPN|dep|5 i|PRON|nsubj|5 will|AUX|aux|5 explain|VERB|ROOT|5

TEXT:      today , for example , I am concerned about the .
BRACKETED: (ROOT be/AUX (npadvmod today/NOUN) (prep for/ADP (pobj example/NOUN)) (nsubj i/PRON) (acomp concern/VERB (prep about/ADP (pobj the/PRON))))
FLAT:      today|NOUN|npadvmod|6 for|ADP|prep|6 example|NOUN|pobj|2 i|PRON|nsubj|6 be|AUX|ROOT|6 concern|VERB|acomp|6 about|ADP|prep|7 the|PRON|pobj|8

TEXT:      however , to support his .
BRACKETED: (ROOT support/VERB (advmod however/ADV) (aux to/PART) (dobj his/PRON))
FLAT:      however|ADV|advmod|3 to|PART|aux|3 support|VERB|ROOT|3 his|PRON|dobj|3

TEXT:      therefore , I believe that we should aim for a postponement and then ultimately .
BRACKETED: (ROOT believe/VERB (advmod therefore/ADV) (nsubj i/PRON) (ccomp aim/VERB (mark that/SCONJ) (nsubj 

# Translators

Because I want to use multiple translation schemes on the same test data, I made a simple `Translator` superclass to make things interchangeable.
It records its `name` and has a `translate()` method that maps English sentences to glosses; each subclass supplies its own `translate()`.

Any translator can read a parse instead of the raw English.
With `parse_format="flat"` or `"bracketed"`, the translator's name gets a `+flat` or `+bracketed` suffix, and `parsed_strings()` and `parsed_tokens()` return each sentence's parse, as a string or as tokens.


In [15]:
class Translator:
    """Base class for everything that maps English sentences to ASL gloss.

    A subclass sets base_name and implements translate().
    With parse_format=None it reads the raw English.
    Otherwise its name gets a "+flat" or "+bracketed" suffix and it can call
    parsed_strings() or parsed_tokens().
    Every translator uses parse_sentences(), so each sentence is parsed only
    once.
    """

    base_name: ClassVar[str]

    # Seconds fit() took; rule-based translators need no training.
    training_seconds: float = 0.0

    def __init__(self, parse_format: str | None = None) -> None:
        """Store the parse format and build the translator's name.

        Args:
            parse_format (str | None): "flat", "bracketed", or None to
                read the raw English.
                Defaults to None.
        """

        self.parse_format: str | None = parse_format

        # e.g. "copy" or "copy+flat", used as the system name in results.
        self.name: str = (
            self.base_name
            if self.parse_format is None
            else f"{self.base_name}+{self.parse_format}"
        )

    def parsed_strings(self, sentences: list[str]) -> pd.Series:
        """The serialized parse of each sentence, in input order."""
        if self.parse_format is None:
            raise ValueError(f"{self.name} was created without a parse_format")
        return parse_sentences(sentences)[self.parse_format]

    def parsed_tokens(self, sentences: list[str]) -> list[list[tuple[str, ...]]]:
        """The parse of each sentence as tokens, in input order.

        Flat parses give tokens in sentence order.
        Bracketed parses give them in tree order: each head before its
        dependents, depth first.
        A text with several sentences gives all their tokens in one list."""
        if self.parse_format is None:
            raise ValueError(f"{self.name} was created without a parse_format")
        return parse_sentences(sentences)[f"{self.parse_format}_tokens"].tolist()

    def translate(self, sentences: list[str]) -> list[str]:
        """Translate English sentences to gloss"""
        raise NotImplementedError

## Baseline (copy to uppercase)

`CopyTranslator` simply returns the input in uppercase.
If a translator does worse than this, it's a problem.

With a parse, it copies the parse's lemmas instead, in sentence order for `flat` and head-before-dependents order for `bracketed`.


In [16]:
class CopyTranslator(Translator):
    """Return the input uppercased, with whitespace normalized.
    With a parse, punctuation is left out."""

    base_name: ClassVar[str] = "copy"

    def translate(self, sentences: list[str]) -> list[str]:
        """Uppercase each sentence, or its parse's lemmas."""

        # split() + join collapses runs of whitespace.
        if self.parse_format is None:
            return [" ".join(sentence.upper().split()) for sentence in sentences]

        # Return lemmas, in the parse format's token order.
        return [
            " ".join(lemma.upper() for lemma, *_ in tokens)
            for tokens in self.parsed_tokens(sentences)
        ]

## Baseline (word substitution)

`WordSubstitutionTranslator` is the rule-based baseline. It lemmatizes word by word, drops articles, "be" (`BASELINE_DROPPED_LEMMAS`) and punctuation, and uppercases the rest.

With a parse, it also uses part-of-speech tags and dependency labels: it drops tokens whose label is in `PARSE_DROPPED_DEPS` and adds the `X-` and `DESC-` prefixes ASLG-PC12 uses.
The bracketed version reads tokens head-before-dependents, so it also tests a tree-based word order.


In [17]:
class WordSubstitutionTranslator(Translator):
    """Baseline: word-by-word lemmatize, drop, and uppercase.

    Lemmas in BASELINE_DROPPED_LEMMAS and punctuation are dropped.
    With a parse it also uses part-of-speech tags and dependency labels:
    it drops tokens whose label is in PARSE_DROPPED_DEPS and adds the X-/DESC-
    prefixes ASLG-PC12 uses (see translate()).
    "bracketed" reads tokens head-before-dependents, so it also tests a
    tree-based word order.
    """

    base_name: ClassVar[str] = "word_substitution"

    def __init__(self, parse_format: str | None = None) -> None:
        """Set up the translator; the plain version loads its own lemmatizer."""
        super().__init__(parse_format)

        # Lemmas only need the tagger, so the parser and NER are disabled.
        # Parse versions use parse_sentences() instead and need no pipeline.
        self.nlp: Language | None = (
            spacy.load(SPACY_MODEL, disable=["parser", "ner"])
            if self.parse_format is None
            else None
        )

    def translate(self, sentences: list[str]) -> list[str]:
        """Gloss each sentence word by word."""

        # Lemmas dropped in every version.
        # "be" covers is/are/was/were/been/being/am and contractions like 's.
        BASELINE_DROPPED_LEMMAS: frozenset[str] = frozenset({"a", "an", "the", "be"})

        # Plain version: lemmatize with spaCy, drop punctuation and
        # BASELINE_DROPPED_LEMMAS, uppercase what is left.
        if self.nlp is not None:
            return [
                " ".join(
                    token.lemma_.upper()
                    for token in doc
                    if not (token.is_punct or token.is_space)
                    and token.lemma_.lower() not in BASELINE_DROPPED_LEMMAS
                )
                for doc in self.nlp.pipe(sentences, batch_size=SPACY_BATCH_SIZE)
            ]

        # Gloss prefixes chosen from the parse: pronouns get X-, adjectives and
        # adverbs get DESC-, and so does negation ("not" -> DESC-NOT).
        # A dependency rule wins over a part-of-speech rule.
        PARSE_PREFIX_BY_POS: dict[str, str] = {
            "PRON": "X",
            "ADJ": "DESC",
            "ADV": "DESC",
        }
        PARSE_PREFIX_BY_DEP: dict[str, str] = {"neg": "DESC"}

        # Tokens with these dependency labels are dropped:
        # "there" in "there is" (expl) and possessive "'s" (case),
        # both of which ASLG-PC12 gloss leaves out.
        PARSE_DROPPED_DEPS: frozenset[str] = frozenset({"expl", "case"})

        # Pronoun lemmas ASLG-PC12 leaves without a prefix
        # (demonstratives, wh-words, quantifiers);
        # personal pronouns like "I", "we", "it" still get X-.
        PARSE_UNPREFIXED_LEMMAS: frozenset[str] = frozenset(
            {
                "this", "that", "these", "those", "what", "which", "who",
                "whom", "whose", "all", "both", "each", "some", "any", "none",
                "something", "anything", "nothing", "everything", "someone",
                "anyone", "everyone", "one", "other", "another", "such",
                "many", "much", "few", "several",
            }
        )  # fmt: skip

        # Parse version: the same drops plus PARSE_DROPPED_DEPS, and a prefix
        # chosen by dependency label first, then by part of speech.
        glosses: list[str] = []
        for tokens in self.parsed_tokens(sentences):
            words: list[str] = []
            for lemma, pos, dep, _ in tokens:
                if (
                    pos in {"PUNCT", "SPACE"}
                    or dep in PARSE_DROPPED_DEPS
                    or lemma in BASELINE_DROPPED_LEMMAS
                ):
                    continue

                # Dependency prefix first; unprefixed pronouns skip the POS one.
                prefix: str | None = PARSE_PREFIX_BY_DEP.get(dep)
                if prefix is None and lemma not in PARSE_UNPREFIXED_LEMMAS:
                    prefix = PARSE_PREFIX_BY_POS.get(pos)
                word: str = lemma.upper()
                words.append(f"{prefix}-{word}" if prefix else word)
            glosses.append(" ".join(words))
        return glosses

## Translator test


`translator_test()` runs translators on three hand-written sentences that exercise articles, "be", tense and a question.
It prints each sentence with every translator's output below it, and returns them as a DataFrame with one column per translator.
Here it runs the plain copy and word-substitution translators.


In [18]:
def translator_test(
    translators: Sequence[Translator], samples: pd.DataFrame | None = None
) -> pd.DataFrame:
    """Run translators on the samples' english column; one output column per
    translator.
    Any other sample columns (e.g. a reference gloss or a note) are kept and
    printed above the outputs.
    Without samples, three built-in sentences are used."""

    # By default, a few sentences that exercise articles, "be", tense and a
    # question.
    if samples is None:
        samples = pd.DataFrame(
            {
                "english": [
                    "The dog ate the bone.",
                    "She is going to the store tomorrow.",
                    "Where did you buy those shoes?",
                ]
            }
        )
    sentences: list[str] = samples["english"].tolist()
    context: list[str] = [column for column in samples.columns if column != "english"]

    # Translate the whole batch with each translator in turn.
    outputs: pd.DataFrame = samples.copy()
    for translator in translators:
        outputs[translator.name] = translator.translate(sentences)

    # Print each sample once, with every translator's output under it.
    for _, row in outputs.iterrows():
        print(row["english"])
        for column in context:
            print(f"  {'[' + column + ']':<28}    {row[column]}")
        for translator in translators:
            print(f"  {translator.name:<28} -> {row[translator.name]}")

        # A blank line separates samples when each has several lines.
        if context:
            print()
    return outputs


# The plain rule-based translators.
translator_samples_df: pd.DataFrame = translator_test(
    [CopyTranslator(), WordSubstitutionTranslator()]
)

# The same translators again, now reading each parse format.
parse_translator_samples_df: pd.DataFrame = translator_test(
    [
        translator_class(parse_format=parse_format)
        for translator_class in (CopyTranslator, WordSubstitutionTranslator)
        for parse_format in ("flat", "bracketed")
    ]
)

The dog ate the bone.
  copy                         -> THE DOG ATE THE BONE.
  word_substitution            -> DOG EAT BONE
She is going to the store tomorrow.
  copy                         -> SHE IS GOING TO THE STORE TOMORROW.
  word_substitution            -> SHE GO TO STORE TOMORROW
Where did you buy those shoes?
  copy                         -> WHERE DID YOU BUY THOSE SHOES?
  word_substitution            -> WHERE DO YOU BUY THOSE SHOE
The dog ate the bone.
  copy+flat                    -> THE DOG EAT THE BONE
  copy+bracketed               -> EAT DOG THE BONE THE
  word_substitution+flat       -> DOG EAT BONE
  word_substitution+bracketed  -> EAT DOG BONE
She is going to the store tomorrow.
  copy+flat                    -> SHE BE GO TO THE STORE TOMORROW
  copy+bracketed               -> GO SHE BE TO STORE THE TOMORROW
  word_substitution+flat       -> X-SHE GO TO STORE TOMORROW
  word_substitution+bracketed  -> GO X-SHE TO STORE TOMORROW
Where did you buy those shoes?
  cop

# Neural translators

Code that all three neural translators (LSTM, Transformer and T5) share.
`TrainingConfig` holds the settings each of them has (epochs, batch size, learning rate, device, input and output length limits, and so on), and `NeuralTranslator` provides:

- `fit()`: training with Adam and gradient clipping, scoring the validation set after each epoch and recording the loss and token accuracy on both sets. Training stops after `EARLY_STOPPING_PATIENCE` epochs without a new best validation loss, and the best epoch's weights are restored. Batches can be filled with sentences of similar length to cut padding.
- `translate()`: greedy decoding, `DECODE_BATCH_SIZE` sentences at a time.

Each model only supplies how to build its network, encode its inputs and targets, compute a batch's loss, and generate.


In [19]:
# kw_only lets each subclass give epochs, batch_size and learning_rate its
# own default, and means configs are always built with keyword arguments.
@dataclass(frozen=True, slots=True, kw_only=True)
class TrainingConfig:
    """Training settings every neural translator's config has.

    The first seven have no default here; each model's config sets them from
    its own section of the settings cell.

    Attributes:
        epochs (int): Passes over the training data.
        batch_size (int): Sentence pairs per training and validation batch.
        learning_rate (float): Adam learning rate.
        device (str | None): Torch device ("cpu", "cuda", "mps"), or None
            for the fastest available.
        bucket_by_length (bool): Fill each training batch with sentences of
            similar length.
        max_source_len (int): Source positions (words or subword tokens)
            kept per input.
        max_target_len (int): Gloss tokens kept per sentence in training;
            also caps generated gloss length.
        decode_batch_size (int): Sentences per batch in translate().
        clip_norm (float): Maximum gradient norm.
        patience (int): Stop after this many epochs in a row without a new
            best validation loss.
        seed (int): Seed for weight initialization, dropout and shuffling.
    """

    epochs: int
    batch_size: int
    learning_rate: float
    device: str | None
    bucket_by_length: bool
    max_source_len: int
    max_target_len: int
    decode_batch_size: int = DECODE_BATCH_SIZE
    clip_norm: float = CLIP_NORM
    patience: int = EARLY_STOPPING_PATIENCE
    seed: int = RANDOM_SEED


class NeuralTranslator[C: TrainingConfig, M: nn.Module](Translator):
    """Training loop and batched decoding shared by the neural translators.

    A subclass sets base_name and implements _build_model(), _sources(),
    _targets(), _batch_loss() and _generate().
    C is its config class and M its network class, so self.config and self.model
    have those types.
    """

    def __init__(
        self,
        config: C,
        *,
        parse_format: str | None = None,
    ) -> None:
        """Store the settings; fit() builds the model.

        Args:
            config (C): Training settings and model hyperparameters, e.g.
                LSTMConfig() or LSTMConfig(epochs=2).
            parse_format (str | None): None for English, or "flat" /
                "bracketed" to read the parse tree instead.
                The name becomes e.g. "lstm", "lstm+flat" or "lstm+bracketed".
                Defaults to None.
        """
        super().__init__(parse_format)
        self.config: C = config

        # The configured device, or else the fastest one available.
        self.device: torch.device = torch.device(
            config.device
            or (
                "cuda"
                if torch.cuda.is_available()
                else "mps" if torch.backends.mps.is_available() else "cpu"
            )
        )

        # Set by fit(); None and 0 until then.
        self.model: M | None = None
        self.best_epoch: int = 0

        # One row per epoch: its number, then the loss and token accuracy on
        # the training and validation data.
        self.history: pd.DataFrame = pd.DataFrame(
            columns=[
                "epoch",
                "training_loss",
                "validation_loss",
                "training_accuracy",
                "validation_accuracy",
            ]
        )

    def fit(
        self,
        training: pd.DataFrame,
        validation: pd.DataFrame | None = None,
        verbose: bool = True,
    ) -> Self:
        """Train on DataFrames with english and gloss columns.

        Builds a fresh model, so calling fit() again starts over.
        With validation data, training stops early once config.patience epochs
        in a row bring no new best validation loss, and the best epoch's weights
        are restored (its number goes to self.best_epoch).
        The losses and token accuracies go to self.history and the wall-clock
        time to self.training_seconds.

        Args:
            training (pd.DataFrame): Training rows with english and gloss
                columns.
            validation (pd.DataFrame | None): Optional validation rows, same
                columns, used for the validation loss and epoch selection.
                Defaults to None.
            verbose (bool): Print the losses and accuracies after each epoch.
                Defaults to True.

        Returns:
            Self: This translator, now trained, so calls can be chained.

        Raises:
            ValueError: If training has no rows.
        """
        if training.empty:
            raise ValueError("No training rows")
        start: float = time.perf_counter()

        # Seed initialization and dropout; shuffling gets its own generator.
        torch.manual_seed(self.config.seed)
        generator: torch.Generator = torch.Generator().manual_seed(self.config.seed)

        # A fresh network (sized from, or loaded for, the training data).
        model: M = self._build_model(training).to(self.device)
        self.model = model
        optimizer = torch.optim.Adam(model.parameters(), lr=self.config.learning_rate)

        # Encode the pairs once, before training; validation is optional.
        training_data = self._encode(training)
        validation_data = (
            None if validation is None or validation.empty else self._encode(validation)
        )

        rows: list[dict[str, float]] = []
        best_loss: float = math.inf
        best_state: dict[str, Tensor] | None = None

        # Epochs in a row without a new best validation loss.
        bad_epochs: int = 0
        for epoch in range(1, self.config.epochs + 1):
            # Train on every training row, then score the validation rows.
            training_loss, training_accuracy = self._run_epoch(
                *training_data, generator, optimizer
            )
            validation_loss, validation_accuracy = (
                (math.nan, math.nan)
                if validation_data is None
                else self._run_epoch(*validation_data, generator)
            )
            rows.append(
                {
                    "epoch": epoch,
                    "training_loss": training_loss,
                    "validation_loss": validation_loss,
                    "training_accuracy": training_accuracy,
                    "validation_accuracy": validation_accuracy,
                }
            )
            if verbose:
                print(
                    f"[{self.name}] epoch {epoch:>3}: "
                    f"training loss {training_loss:.3f}, "
                    f"validation loss {validation_loss:.3f}, "
                    f"training accuracy {training_accuracy:.3f}, "
                    f"validation accuracy {validation_accuracy:.3f}"
                )

            # A new best validation loss: copy its weights and reset the count.
            # NaN never compares lower, so without validation nothing is
            # copied and training never stops early.
            if validation_loss < best_loss:
                best_loss = validation_loss
                self.best_epoch = epoch
                best_state = {
                    k: v.detach().clone() for k, v in model.state_dict().items()
                }
                bad_epochs = 0

            # Otherwise count the epoch; patience bad epochs in a row stop it.
            elif validation_data is not None:
                bad_epochs += 1
                if bad_epochs >= self.config.patience:
                    if verbose:
                        print(
                            f"[{self.name}] stopped early; best epoch was {self.best_epoch}"
                        )
                    break

        # Roll back to the best validation epoch (without validation data,
        # keep the last epoch).
        if best_state is not None:
            model.load_state_dict(best_state)
        else:
            self.best_epoch = len(rows)
        self.history = pd.DataFrame(rows)

        # Everything from building the model to restoring the best weights.
        self.training_seconds = time.perf_counter() - start
        return self

    def _encode(self, df: pd.DataFrame) -> tuple[list[Any], list[Any]]:
        """The model inputs for each pair in df.

        Args:
            df (pd.DataFrame): Rows with english and gloss columns.

        Returns:
            tuple[list[Any], list[Any]]: _sources() of the English and
            _targets() of the glosses, in row order.
        """
        return (
            self._sources(df["english"].tolist()),
            self._targets(df["gloss"].tolist()),
        )

    def _run_epoch(
        self,
        sources: list[Any],
        targets: list[Any],
        generator: torch.Generator,
        optimizer: torch.optim.Optimizer | None = None,
    ) -> tuple[float, float]:
        """One pass over the data: training with an optimizer, else scoring.

        Args:
            sources (list[Any]): Encoded inputs from _sources().
            targets (list[Any]): Matching encoded glosses from _targets().
            generator (torch.Generator): Shuffles the rows when training.
            optimizer (torch.optim.Optimizer | None): Updates the weights after
                each batch; None only scores.
                Defaults to None.

        Returns:
            tuple[float, float]: Mean loss per gold target token, and the
            share of gold target tokens the model ranked first (token
            accuracy, 0 to 1).
            Both use teacher forcing: each position sees the gold tokens
            before it, not the model's own output.
        """

        # Batches per length-sorted group when bucketing.
        BUCKET_BATCHES: int = 50

        assert self.model is not None
        training: bool = optimizer is not None
        size: int = self.config.batch_size

        # Training mode turns dropout on and visits the rows in a new random
        # order each epoch; scoring goes in order.
        self.model.train(training)
        order: list[int] = (
            torch.randperm(len(sources), generator=generator).tolist()
            if training
            else list(range(len(sources)))
        )
        batches: list[list[int]] = [
            order[start : start + size] for start in range(0, len(order), size)
        ]

        # Bucketing: sort each group of shuffled rows by input length and cut
        # it into batches, so a batch needs little padding; then shuffle the
        # batches, so lengths still come in random order.
        if training and self.config.bucket_by_length:
            batches = []
            for start in range(0, len(order), size * BUCKET_BATCHES):
                group: list[int] = sorted(
                    order[start : start + size * BUCKET_BATCHES],
                    key=lambda i: len(sources[i]),
                )
                batches += [group[k : k + size] for k in range(0, len(group), size)]
            batches = [
                batches[b]
                for b in torch.randperm(len(batches), generator=generator).tolist()
            ]
        total, tokens, correct = 0.0, 0, 0

        # Gradients are only needed when training.
        with torch.set_grad_enabled(training):
            for batch in batches:
                loss, count, right = self._batch_loss(
                    [sources[i] for i in batch], [targets[i] for i in batch]
                )

                # Backward pass, clip gradients, update weights.
                if optimizer is not None:
                    optimizer.zero_grad()
                    loss.backward()
                    nn.utils.clip_grad_norm_(
                        self.model.parameters(), self.config.clip_norm
                    )
                    optimizer.step()

                # Weight each batch mean by its token count for the epoch mean.
                total += loss.item() * count
                tokens += count
                correct += right
        return total / max(tokens, 1), correct / max(tokens, 1)

    def translate(self, sentences: list[str]) -> list[str]:
        """Translate English sentences to gloss with greedy decoding.

        Args:
            sentences (list[str]): English sentences.

        Returns:
            list[str]: One space-separated gloss per sentence, in input
            order.

        Raises:
            RuntimeError: If fit() has not been called.

        Example:
            >>> training = pd.DataFrame({
            ...     "english": ["I eat an orange", "you eat an apple"],
            ...     "gloss": ["X-I EAT ORANGE", "X-YOU EAT APPLE"],
            ... })
            >>> lstm = LSTMTranslator(LSTMConfig(epochs=2))
            >>> _ = lstm.fit(training, verbose=False)
            >>> # The gloss itself depends on training.
            >>> len(lstm.translate(["you eat"]))
            1
        """
        if self.model is None:
            raise RuntimeError(f"{self.name}: call fit() before translate()")

        # Evaluation mode turns dropout off; decoding needs no gradients.
        self.model.eval()
        sources: list[Any] = self._sources(sentences)
        outputs: list[str] = []
        step: int = self.config.decode_batch_size
        with torch.no_grad():
            # Decode in batches; output order follows input order.
            for start in range(0, len(sources), step):
                outputs.extend(self._generate(sources[start : start + step]))
        return outputs

    # The methods below are what each subclass provides.

    def _build_model(self, training: pd.DataFrame) -> M:
        """A new network, ready to train.

        Args:
            training (pd.DataFrame): The training rows, for anything sized
                from the data (e.g. vocabularies).

        Returns:
            M: The untrained (or pretrained) network.
        """
        raise NotImplementedError

    def _sources(self, sentences: list[str]) -> list[Any]:
        """The encoded model input for each English sentence.

        Args:
            sentences (list[str]): English sentences.

        Returns:
            list[Any]: One encoded input per sentence.
        """
        raise NotImplementedError

    def _targets(self, glosses: list[str]) -> list[Any]:
        """The encoded training target for each gloss.

        Args:
            glosses (list[str]): Gold glosses.

        Returns:
            list[Any]: One encoded target per gloss.
        """
        raise NotImplementedError

    def _batch_loss(
        self, sources: list[Any], targets: list[Any]
    ) -> tuple[Tensor, int, int]:
        """Mean token loss for one batch, how many tokens it covers, and how
        many of those the model got right.

        Args:
            sources (list[Any]): Encoded inputs from _sources().
            targets (list[Any]): Matching encoded glosses from _targets().

        Returns:
            tuple[Tensor, int, int]: The scalar loss (still attached to the
            graph), the number of gold target tokens scored, and how many of
            them were the model's top prediction.
        """
        raise NotImplementedError

    def _generate(self, sources: list[Any]) -> list[str]:
        """Decode one batch of encoded inputs into glosses.

        Args:
            sources (list[Any]): Encoded inputs from _sources().

        Returns:
            list[str]: One gloss per input, in order.
        """
        raise NotImplementedError

### Vocabularies

`Vocabulary` maps tokens to ids for the from-scratch models (LSTM and Transformer).
Every vocabulary starts with the special tokens `<pad>`, `<unk>`, `<bos>` and `<eos>`, so their ids (`PAD_ID` to `EOS_ID`) are the same in every vocabulary.
Source tokens seen fewer than `SEQ2SEQ_MIN_FREQ` times in training become `<unk>`; the target vocabulary keeps every gloss token.


In [20]:
# Reserved tokens take the first ids in every vocabulary, in this order.
SPECIAL_TOKENS: tuple[str, ...] = ("<pad>", "<unk>", "<bos>", "<eos>")
PAD_ID, UNK_ID, BOS_ID, EOS_ID = range(len(SPECIAL_TOKENS))


class Vocabulary:
    """Map tokens to integer ids; the first ids are the special tokens.

    Example:
        >>> vocab = Vocabulary("EAT ORANGE EAT".split())
        >>> vocab.itos
        ['<pad>', '<unk>', '<bos>', '<eos>', 'EAT', 'ORANGE']
        >>> vocab.id("EAT"), vocab.id("APPLE")
        (4, 1)
    """

    def __init__(self, tokens: Iterable[str], min_freq: int = 1) -> None:
        """Count the tokens and assign ids to the frequent ones.

        Args:
            tokens (Iterable[str]): Every token occurrence in the training
                data; repeats are counted.
            min_freq (int): Tokens seen fewer times map to <unk>.
                Defaults to 1 (keep everything).
        """
        counts: Counter[str] = Counter(tokens)

        # Most frequent first, ties alphabetical, so ids are reproducible.
        kept: list[str] = sorted(
            (t for t, c in counts.items() if c >= min_freq and t not in SPECIAL_TOKENS),
            key=lambda t: (-counts[t], t),
        )

        # Specials first, so PAD_ID ... EOS_ID hold in every vocabulary.
        self.itos: list[str] = [*SPECIAL_TOKENS, *kept]
        self.stoi: dict[str, int] = {token: i for i, token in enumerate(self.itos)}

    def __len__(self) -> int:
        """Number of ids, special tokens included."""
        return len(self.itos)

    def id(self, token: str) -> int:
        """Look up the id of one token."""
        return self.stoi.get(token, UNK_ID)

### Seq2Seq

`Seq2SeqModel` is the base class for the LSTM and Transformer networks.
It embeds each source factor separately and concatenates the embeddings (for the flat parse: lemma, POS, dependency label and head), and provides `forward()`, which scores the gold gloss given the source for training (teacher forcing).
Each subclass supplies its own `encode()`, which reads the source once, and `decode()`, which scores a target prefix.
Greedy decoding in `Seq2SeqTranslator` calls these two directly.


In [21]:
class Seq2SeqModel(nn.Module):
    """Base class for the LSTM and Transformer networks.

    It embeds the factored source and implements forward() on top of two
    methods each subclass provides: encode(), which reads the source once,
    and decode(), which scores a target prefix.
    """

    def __init__(
        self, source_vocab_sizes: Sequence[int], word_dim: int, factor_dim: int
    ) -> None:
        """Create one embedding per source factor.

        Args:
            source_vocab_sizes (Sequence[int]): Vocabulary size of each source
                factor; the first is the word, the rest are parse features.
            word_dim (int): Size of the word embedding.
            factor_dim (int): Size of each extra factor's embedding.
        """
        super().__init__()

        # The first factor is the word; extra factors get smaller embeddings.
        self.source_dims: list[int] = [word_dim] + [factor_dim] * (
            len(source_vocab_sizes) - 1
        )
        self.source_embeddings = nn.ModuleList(
            nn.Embedding(size, dim, padding_idx=PAD_ID)
            for size, dim in zip(source_vocab_sizes, self.source_dims, strict=True)
        )

    def embed_source(self, source: Tensor) -> Tensor:
        """Embed each factor separately and concatenate the embeddings."""
        return torch.cat(
            [embed(source[..., i]) for i, embed in enumerate(self.source_embeddings)],
            dim=-1,
        )

    def encode(self, source: Tensor, lengths: Tensor) -> tuple[Tensor, ...]:
        """Read the source once; subclasses override this."""
        raise NotImplementedError

    def decode(self, target_in: Tensor, encoded: tuple[Tensor, ...]) -> Tensor:
        """Score every position of a target prefix; subclasses override this."""
        raise NotImplementedError

    def forward(self, source: Tensor, lengths: Tensor, target_in: Tensor) -> Tensor:
        """Teacher-forced logits for a batch (used in training)."""
        return self.decode(target_in, self.encode(source, lengths))

#### Seq2SeqTranslator

`Seq2SeqTranslator` adds what the LSTM and Transformer share because both are trained from scratch: building the source and target vocabularies from the training rows, turning sentences into source features, padding batches, and greedy decoding.
Raw English and the bracketed parse give one factor per position, and the flat parse gives four (lemma, POS, dependency label, head).
`BRACKETED_TOKEN` splits a bracketed parse into tokens such as `(nsubj`, `i/PRON` and `)`, so the encoder sees the tree structure directly.
Greedy decoding encodes each batch once, then appends the most likely next token to every row until all rows have produced `<eos>` or the length limit (`max_target_len`) is reached.
Each subclass names its network class in `network`.


In [22]:
class Seq2SeqTranslator[C: TrainingConfig](NeuralTranslator[C, Seq2SeqModel]):
    """Vocabularies, source features and decoding for a from-scratch network.

    A subclass sets base_name and network, the Seq2SeqModel class it trains;
    network is called with (source vocabulary sizes, target vocabulary size,
    config).
    """

    network: ClassVar[Callable[[list[int], int, Any], Seq2SeqModel]]

    @property
    def num_factors(self) -> int:
        """Number of factors per source position.

        Returns:
            int: 4 for the flat parse (lemma, POS, dep, head), otherwise 1.
        """
        return 4 if self.parse_format == "flat" else 1

    def _features(self, sentences: list[str]) -> list[list[tuple[str, ...]]]:
        """The source-side factors for each sentence.

        Args:
            sentences (list[str]): English sentences.

        Returns:
            list[list[tuple[str, ...]]]: One tuple of factors per source
            position, e.g. ("eat",) or ("eat", "VERB", "ROOT", "1"), each of
            length num_factors.
        """

        # Plain English: normalized whitespace tokens, one factor each.
        if self.parse_format is None:
            return [[(token,) for token in normalize(s).split()] for s in sentences]

        # Flat parse: the (lemma, POS, dep, head index) tuples as they are.
        if self.parse_format == "flat":
            return self.parsed_tokens(sentences)

        # "(dep", ")", or any other run of non-space, non-bracket characters.
        BRACKETED_TOKEN: re.Pattern[str] = re.compile(r"\(\S+|\)|[^\s()]+")

        # Bracketed parse: brackets and words as one token stream.
        return [
            [(token,) for token in BRACKETED_TOKEN.findall(text)]
            for text in self.parsed_strings(sentences)
        ]

    def _build_model(self, training: pd.DataFrame) -> Seq2SeqModel:
        """Build the vocabularies from the training rows, then the network.

        Args:
            training (pd.DataFrame): Training rows with english and gloss
                columns.

        Returns:
            Seq2SeqModel: A new, untrained network sized to the vocabularies.
        """
        features: list[list[tuple[str, ...]]] = self._features(
            training["english"].tolist()
        )

        # One vocabulary per factor, built from that factor's column.
        self.source_vocabs: list[Vocabulary] = [
            Vocabulary(
                (position[i] for sentence in features for position in sentence),
                SEQ2SEQ_MIN_FREQ,
            )
            for i in range(self.num_factors)
        ]

        # The target vocabulary keeps every gloss token (no min_freq).
        self.target_vocab: Vocabulary = Vocabulary(
            token for gloss in training["gloss"] for token in gloss.split()
        )

        # type(self) reads network as a plain class attribute (through self,
        # type checkers would treat it as a method).
        return type(self).network(
            [len(vocab) for vocab in self.source_vocabs],
            len(self.target_vocab),
            self.config,
        )

    def _sources(self, sentences: list[str]) -> list[Tensor]:
        """Source ids per sentence, shape (src_len, factors), truncated to
        max_source_len.

        Args:
            sentences (list[str]): English sentences.

        Returns:
            list[Tensor]: One id tensor per sentence; an empty input becomes
            a single <unk>.
        """
        sources: list[Tensor] = []
        for features in self._features(sentences):
            # Look up each factor in its own vocabulary.
            rows: list[list[int]] = [
                [
                    vocab.id(factor)
                    for vocab, factor in zip(self.source_vocabs, position, strict=True)
                ]
                for position in features[: self.config.max_source_len]
            ]

            # pack_padded_sequence rejects zero-length rows, hence the <unk>.
            sources.append(
                torch.tensor(rows or [[UNK_ID] * self.num_factors], dtype=torch.long)
            )
        return sources

    def _targets(self, glosses: list[str]) -> list[Tensor]:
        """<bos> gloss <eos> ids, truncated to max_target_len gloss tokens.

        Args:
            glosses (list[str]): Space-separated gold glosses.

        Returns:
            list[Tensor]: One id tensor per gloss, shape (tgt_len + 2,).
        """
        targets: list[Tensor] = []
        for gloss in glosses:
            ids: list[int] = [
                self.target_vocab.id(token)
                for token in gloss.split()[: self.config.max_target_len]
            ]
            targets.append(torch.tensor([BOS_ID, *ids, EOS_ID], dtype=torch.long))
        return targets

    def _pad_sources(self, sources: list[Tensor]) -> tuple[Tensor, Tensor]:
        """Pad a batch of source tensors to a common length.

        Args:
            sources (list[Tensor]): Source ids, each shape (src_len, factors).

        Returns:
            tuple[Tensor, Tensor]: Padded ids on self.device, shape
            (batch, max_src_len, factors), and lengths on the CPU, shape
            (batch,).
        """
        lengths: Tensor = torch.tensor([len(s) for s in sources])
        padded: Tensor = pad_sequence(sources, batch_first=True, padding_value=PAD_ID)
        return padded.to(self.device), lengths

    def _batch_loss(
        self, sources: list[Tensor], targets: list[Tensor]
    ) -> tuple[Tensor, int, int]:
        """Mean token loss for one batch, the number of gold tokens, and how
        many of them the model predicted.

        Args:
            sources (list[Tensor]): Source ids, each shape (src_len, factors).
            targets (list[Tensor]): Target ids with <bos> and <eos>, each
                shape (tgt_len + 2,).

        Returns:
            tuple[Tensor, int, int]: The scalar loss (still attached to the
            graph), the number of non-padding gold tokens, and how many of
            them were the model's top prediction.
        """
        assert self.model is not None
        source, lengths = self._pad_sources(sources)
        target: Tensor = pad_sequence(
            targets, batch_first=True, padding_value=PAD_ID
        ).to(self.device)

        # Teacher forcing: input is target without its last token, gold is
        # target without <bos>, so position t predicts token t + 1.
        logits: Tensor = self.model(source, lengths, target[:, :-1])
        gold: Tensor = target[:, 1:]

        # Flatten (batch, len, vocab) to (batch * len, vocab); padding is
        # left out of the mean.
        loss: Tensor = nn.functional.cross_entropy(
            logits.reshape(-1, logits.size(-1)), gold.reshape(-1), ignore_index=PAD_ID
        )
        real: Tensor = gold != PAD_ID

        # Correct: a real gold token that is also the highest-scoring one.
        right: int = int(((logits.argmax(dim=-1) == gold) & real).sum())
        return loss, int(real.sum()), right

    def _generate(self, sources: list[Tensor]) -> list[str]:
        """Greedy-decode one batch and turn the ids back into glosses.

        Args:
            sources (list[Tensor]): Source ids, each shape (src_len, factors).

        Returns:
            list[str]: One gloss per source, with at most max_target_len + 1
            tokens; it may contain "<unk>".
        """
        assert self.model is not None
        source, lengths = self._pad_sources(sources)

        # Encode once; every step reuses it.
        encoded: tuple[Tensor, ...] = self.model.encode(source, lengths)

        # Every row starts from <bos>.
        generated: Tensor = torch.full(
            (source.size(0), 1), BOS_ID, dtype=torch.long, device=source.device
        )
        finished: Tensor = torch.zeros(
            source.size(0), dtype=torch.bool, device=source.device
        )

        # One extra step leaves room for <eos> after a full-length gloss.
        for _ in range(self.config.max_target_len + 1):
            # Greedy: append the most likely next token for each row.
            # The decoder re-reads the whole prefix each step (no caching).
            token: Tensor = self.model.decode(generated, encoded)[:, -1].argmax(
                dim=-1, keepdim=True
            )
            generated = torch.cat([generated, token], dim=1)

            # Stop early once every row has emitted <eos> at least once.
            finished |= token.squeeze(1) == EOS_ID
            if finished.all():
                break

        # Each row's tokens after the <bos> column, up to its first <eos>,
        # without <pad> or <bos>.
        # Rows that finished early kept generating until all were done.
        return [
            " ".join(
                self.target_vocab.itos[i]
                for i in takewhile(lambda i: i != EOS_ID, ids[1:])
                if i not in (PAD_ID, BOS_ID)
            )
            for ids in generated.tolist()
        ]

### LSTM

The LSTM model: a bidirectional LSTM encoder and an LSTM decoder with Luong ("general") attention, trained with teacher forcing.
`LSTMConfig` takes its defaults from the settings cell; training, vocabularies and decoding come from `Seq2SeqTranslator`.


In [23]:
"""PyTorch LSTM encoder-decoder translator.

Architecture: bidirectional LSTM encoder, LSTM decoder with Luong
("general") attention, teacher forcing during training, and greedy decoding.
Training, vocabularies and decoding come from Seq2SeqTranslator (previous
cell).
"""

@dataclass(frozen=True, slots=True, kw_only=True)
class LSTMConfig(TrainingConfig):
    """LSTM hyperparameters; the defaults come from the settings cell.

    Attributes:
        embed_dim (int): Size of the word embeddings, source and target.
        factor_dim (int): Size of each extra source factor's embedding.
        hidden_dim (int): LSTM hidden size per direction.
        num_layers (int): Stacked LSTM layers in encoder and decoder.
        dropout (float): Dropout on embeddings, between LSTM layers (when
            there are several), and before the output layer.

    Example:
        >>> LSTMConfig(epochs=2, hidden_dim=64).hidden_dim
        64
    """

    embed_dim: int = LSTM_EMBED_DIM
    factor_dim: int = LSTM_FACTOR_DIM
    hidden_dim: int = LSTM_HIDDEN_DIM
    num_layers: int = LSTM_NUM_LAYERS
    dropout: float = LSTM_DROPOUT

    # Training defaults for the TrainingConfig fields without one.
    epochs: int = LSTM_EPOCHS
    batch_size: int = LSTM_BATCH_SIZE
    learning_rate: float = LSTM_LEARNING_RATE
    device: str | None = LSTM_DEVICE
    max_source_len: int = SEQ2SEQ_MAX_SOURCE_LEN
    max_target_len: int = SEQ2SEQ_MAX_TARGET_LEN
    bucket_by_length: bool = SEQ2SEQ_BUCKET_BY_LENGTH


class Seq2SeqLSTM(Seq2SeqModel):
    """Bidirectional LSTM encoder and an LSTM decoder with Luong attention."""

    def __init__(
        self,
        source_vocab_sizes: Sequence[int],
        target_vocab_size: int,
        config: LSTMConfig,
    ) -> None:
        """Create the embeddings, encoder, decoder, attention and output layers.

        Args:
            source_vocab_sizes (Sequence[int]): Vocabulary size of each source
                factor; the first is the word, the rest are parse features.
            target_vocab_size (int): Number of gloss ids the decoder predicts.
            config (LSTMConfig): The layer sizes and dropout.
        """
        super().__init__(source_vocab_sizes, config.embed_dim, config.factor_dim)
        hidden: int = config.hidden_dim

        # nn.LSTM only applies dropout between stacked layers.
        between_layers: float = config.dropout if config.num_layers > 1 else 0.0

        # Encoder reads the concatenated factor embeddings in both directions.
        self.encoder = nn.LSTM(
            sum(self.source_dims),
            hidden,
            config.num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=between_layers,
        )

        # Squeeze the two encoder directions into the decoder's starting state.
        self.bridge_h = nn.Linear(2 * hidden, hidden)
        self.bridge_c = nn.Linear(2 * hidden, hidden)

        # Decoder side: gloss embeddings and a one-direction LSTM.
        self.target_embedding = nn.Embedding(
            target_vocab_size, config.embed_dim, padding_idx=PAD_ID
        )
        self.decoder = nn.LSTM(
            config.embed_dim,
            hidden,
            config.num_layers,
            batch_first=True,
            dropout=between_layers,
        )

        # Luong "general" attention: score = decoder_output . (W memory).
        self.attention_key = nn.Linear(2 * hidden, hidden, bias=False)

        # Mixes [decoder output; context] into the attentional hidden state.
        self.attention_out = nn.Linear(3 * hidden, hidden)

        # Projects the attentional state to target-vocabulary logits.
        self.generator = nn.Linear(hidden, target_vocab_size)
        self.dropout = nn.Dropout(config.dropout)

    @staticmethod
    def _bridge(state: Tensor, layer: nn.Linear) -> Tensor:
        """(layers * 2, batch, hidden) -> (layers, batch, hidden).

        Args:
            state (Tensor): Final encoder h or c, shape
                (layers * 2, batch, hidden), directions interleaved per layer.
            layer (nn.Linear): Projection from 2 * hidden to hidden.

        Returns:
            Tensor: Decoder initial state, shape (layers, batch, hidden).
        """
        layers_x2, batch, hidden = state.shape

        # Separate the forward (0) and backward (1) direction of each layer.
        both: Tensor = state.view(layers_x2 // 2, 2, batch, hidden)
        return torch.tanh(layer(torch.cat([both[:, 0], both[:, 1]], dim=-1)))

    def encode(self, source: Tensor, lengths: Tensor) -> tuple[Tensor, ...]:
        """Run the encoder and prepare attention and the decoder's start state.

        Args:
            source (Tensor): Source ids, shape (batch, src_len, factors),
                padded with PAD_ID.
            lengths (Tensor): Unpadded length of each row, shape (batch,).

        Returns:
            tuple[Tensor, ...]: memory, shape (batch, src_len, 2 * hidden);
            attention keys, shape (batch, src_len, hidden); the source mask,
            True at real tokens, shape (batch, src_len); and the decoder's
            initial h and c, each shape (layers, batch, hidden).
        """

        # Packing makes the LSTM skip padding, so the final states are
        # taken at each row's real last token.
        packed = pack_padded_sequence(
            self.dropout(self.embed_source(source)),
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        packed_memory, (h, c) = self.encoder(packed)

        # Unpack back to the full padded width so it lines up with the mask.
        memory, _ = pad_packed_sequence(
            packed_memory, batch_first=True, total_length=source.size(1)
        )

        # Keys are computed once here and reused at every decoding step; the
        # mask comes from the word factor, where PAD_ID marks padding.
        return (
            memory,
            self.attention_key(memory),
            source[..., 0] != PAD_ID,
            self._bridge(h, self.bridge_h),
            self._bridge(c, self.bridge_c),
        )

    def decode(self, target_in: Tensor, encoded: tuple[Tensor, ...]) -> Tensor:
        """Logits for every position of target_in, attending to the source.

        Args:
            target_in (Tensor): Decoder input ids starting with <bos>, shape
                (batch, tgt_len).
            encoded (tuple[Tensor, ...]): The output of encode().

        Returns:
            Tensor: Logits, shape (batch, tgt_len, target_vocab_size).
        """
        memory, keys, source_mask, h, c = encoded

        # Run the decoder LSTM over the target inputs from the bridged state.
        output, _ = self.decoder(self.dropout(self.target_embedding(target_in)), (h, c))

        # Attention: score every source position for every target position,
        # shape (batch, tgt_len, src_len).
        scores: Tensor = output @ keys.transpose(1, 2)

        # Padding gets -inf so softmax gives it zero weight.
        scores = scores.masked_fill(~source_mask.unsqueeze(1), float("-inf"))

        # Context: attention-weighted sum of encoder outputs.
        context: Tensor = scores.softmax(dim=-1) @ memory

        # Combine decoder output and context, then project to the vocabulary.
        combined: Tensor = torch.tanh(
            self.attention_out(torch.cat([output, context], dim=-1))
        )
        return self.generator(self.dropout(combined))


class LSTMTranslator(Seq2SeqTranslator[LSTMConfig]):
    """Seq2seq LSTM translator: LSTMTranslator(LSTMConfig(), parse_format=...).

    Call fit() before translate().
    """

    base_name: ClassVar[str] = "lstm"
    network = Seq2SeqLSTM

### Transformer

The start of the Transformer model, which shares everything except its network with the LSTM.
`TransformerConfig` takes its defaults from the settings cell.


In [24]:
"""PyTorch Transformer encoder-decoder translator.

It shares everything except the network with the LSTM (see
Seq2SeqTranslator): the same parse_format options, vocabularies, training
loop, validation-based epoch selection, and greedy decoding.

Architecture: the standard encoder-decoder from Vaswani et al. (2017),
"Attention Is All You Need", built from torch.nn layers, with sinusoidal
positions and pre-layer-norm blocks, which train stably without a learning rate
warmup.
For the flat parse, the factor embeddings (lemma, POS, dep, head) are
concatenated and projected to d_model.
"""

@dataclass(frozen=True, slots=True, kw_only=True)
class TransformerConfig(TrainingConfig):
    """Transformer hyperparameters; the defaults come from the settings cell.

    Attributes:
        d_model (int): Model width (word embeddings and hidden states).
        factor_dim (int): Size of each extra source factor's embedding.
        num_heads (int): Attention heads; must divide d_model.
        encoder_layers (int): Number of encoder layers.
        decoder_layers (int): Number of decoder layers.
        ff_dim (int): Feed-forward width inside each layer.
        dropout (float): Dropout used throughout the layers.

    Example:
        >>> TransformerConfig(epochs=2, num_heads=8).num_heads
        8
    """

    d_model: int = TRANSFORMER_D_MODEL
    factor_dim: int = TRANSFORMER_FACTOR_DIM
    num_heads: int = TRANSFORMER_NUM_HEADS
    encoder_layers: int = TRANSFORMER_ENCODER_LAYERS
    decoder_layers: int = TRANSFORMER_DECODER_LAYERS
    ff_dim: int = TRANSFORMER_FF_DIM
    dropout: float = TRANSFORMER_DROPOUT

    # Training defaults for the TrainingConfig fields without one.
    epochs: int = TRANSFORMER_EPOCHS
    batch_size: int = TRANSFORMER_BATCH_SIZE
    learning_rate: float = TRANSFORMER_LEARNING_RATE
    device: str | None = TRANSFORMER_DEVICE
    max_source_len: int = SEQ2SEQ_MAX_SOURCE_LEN
    max_target_len: int = SEQ2SEQ_MAX_TARGET_LEN
    bucket_by_length: bool = SEQ2SEQ_BUCKET_BY_LENGTH

#### S2S Transformer

`Seq2SeqTransformer` is the standard Transformer encoder-decoder (Vaswani et al., 2017), built from `torch.nn` layers with pre-layer-norm blocks, which train stably without a learning-rate warm-up.
For the flat parse, the factor embeddings are concatenated and projected to `d_model`.
Fixed sine/cosine position signals are computed once and added to the source and target embeddings (`_add_positions()`).
`TransformerTranslator` plugs it into `Seq2SeqTranslator`.


In [25]:
class Seq2SeqTransformer(Seq2SeqModel):
    """Transformer encoder and decoder over factored source embeddings."""

    # Set by register_buffer() in __init__; declared so type checkers know it
    # is a Tensor rather than a submodule.
    position_table: Tensor

    def __init__(
        self,
        source_vocab_sizes: Sequence[int],
        target_vocab_size: int,
        config: TransformerConfig,
    ) -> None:
        """Create the embeddings, encoder and decoder stacks, and output layer.

        Args:
            source_vocab_sizes (Sequence[int]): Vocabulary size of each source
                factor; the first is the word, the rest are parse features.
            target_vocab_size (int): Number of gloss ids the decoder predicts.
            config (TransformerConfig): Layer sizes, heads and dropout, plus
                the length limits that size the positional table.

        Raises:
            ValueError: If d_model is not divisible by num_heads.

        Example:
            >>> config = TransformerConfig(d_model=16)
            >>> model = Seq2SeqTransformer([10], 8, config)
        """
        d_model: int = config.d_model
        if d_model % config.num_heads:
            raise ValueError("d_model must be divisible by num_heads")
        super().__init__(source_vocab_sizes, d_model, config.factor_dim)
        self.d_model: int = d_model

        # With extra factors, project the concatenation back to d_model.
        self.source_projection: nn.Module = (
            nn.Linear(sum(self.source_dims), d_model)
            if len(self.source_dims) > 1
            else nn.Identity()
        )
        self.target_embedding = nn.Embedding(
            target_vocab_size, d_model, padding_idx=PAD_ID
        )

        # One table of sinusoidal position signals for source and target.
        # Targets add <bos> and <eos>, and greedy decoding adds one more step.
        max_len: int = max(config.max_source_len, config.max_target_len + 3)
        positions: Tensor = torch.arange(max_len).unsqueeze(1)

        # Geometric series of wavelengths, 1 / 10000^(2i / d_model).
        frequencies: Tensor = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        # Sine on even dimensions, cosine on odd ones.
        # The slice keeps the cosine half the right width when d_model is odd.
        table: Tensor = torch.zeros(max_len, d_model)
        table[:, 0::2] = torch.sin(positions * frequencies)
        table[:, 1::2] = torch.cos(positions * frequencies[: d_model // 2])

        # A buffer moves with .to(device) but is not trained.
        self.register_buffer("position_table", table, persistent=False)
        self.dropout = nn.Dropout(config.dropout)

        # Pre-norm (norm_first) layers need a final LayerNorm on each stack.
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model,
                config.num_heads,
                config.ff_dim,
                config.dropout,
                batch_first=True,
                norm_first=True,
            ),
            config.encoder_layers,
            norm=nn.LayerNorm(d_model),
            # Nested tensors are a speed-up that pre-norm layers can't use.
            enable_nested_tensor=False,
        )
        self.decoder = nn.TransformerDecoder(
            nn.TransformerDecoderLayer(
                d_model,
                config.num_heads,
                config.ff_dim,
                config.dropout,
                batch_first=True,
                norm_first=True,
            ),
            config.decoder_layers,
            norm=nn.LayerNorm(d_model),
        )

        # Projects decoder outputs to target-vocabulary logits.
        self.generator = nn.Linear(d_model, target_vocab_size)

    def _add_positions(self, embedded: Tensor) -> Tensor:
        """Scale embeddings by sqrt(d_model), as in Vaswani et al., then add
        the position signals and apply dropout.

        Args:
            embedded (Tensor): Embeddings, shape (batch, len, d_model).

        Returns:
            Tensor: The same shape, ready for the encoder or decoder.
        """
        return self.dropout(
            embedded * math.sqrt(self.d_model) + self.position_table[: embedded.size(1)]
        )

    def encode(self, source: Tensor, lengths: Tensor) -> tuple[Tensor, ...]:
        """Run the encoder over the embedded source.

        Args:
            source (Tensor): Source ids, shape (batch, src_len, factors),
                padded with PAD_ID.
            lengths (Tensor): Unused: padding is read from the PAD ids.

        Returns:
            tuple[Tensor, ...]: Encoder memory, shape
            (batch, src_len, d_model), and the padding mask, shape
            (batch, src_len), True at padding.
        """

        # Embed the factors, project to d_model, and add positions.
        x: Tensor = self._add_positions(
            self.source_projection(self.embed_source(source))
        )

        # True where there is no token
        padding: Tensor = source[..., 0] == PAD_ID
        return self.encoder(x, src_key_padding_mask=padding), padding

    def decode(self, target_in: Tensor, encoded: tuple[Tensor, ...]) -> Tensor:
        """Logits for every position; each position sees only earlier targets.

        Args:
            target_in (Tensor): Decoder input ids starting with <bos>, shape
                (batch, tgt_len).
            encoded (tuple[Tensor, ...]): The output of encode().

        Returns:
            Tensor: Logits, shape (batch, tgt_len, target_vocab_size).
        """
        memory, source_padding = encoded
        length: int = target_in.size(1)

        # True above the diagonal blocks attention to later positions.
        causal: Tensor = torch.triu(
            torch.ones(length, length, dtype=torch.bool, device=target_in.device),
            diagonal=1,
        )
        x: Tensor = self._add_positions(self.target_embedding(target_in))

        # Masked self-attention over the target, cross-attention over the
        # source memory; padding on both sides is ignored.
        output: Tensor = self.decoder(
            x,
            memory,
            tgt_mask=causal,
            tgt_key_padding_mask=target_in == PAD_ID,
            memory_key_padding_mask=source_padding,
        )
        return self.generator(output)


class TransformerTranslator(Seq2SeqTranslator[TransformerConfig]):
    """Seq2seq Transformer translator:
    TransformerTranslator(TransformerConfig(), parse_format=...).

    Call fit() before translate().
    """

    base_name: ClassVar[str] = "transformer"
    network = Seq2SeqTransformer

### T5

`T5Translator` fine-tunes a pretrained T5 (`T5_MODEL`, flan-t5-small) instead of training from scratch, so it starts out already knowing English.
Its subword tokenizer replaces the word vocabularies, so an unseen word is spelled out in pieces instead of becoming `<unk>`.
The input is `T5_PREFIX` followed by the English, or by the flat or bracketed parse string.
Training and decoding come from `NeuralTranslator`.
The first run downloads the checkpoint (about 300 MB) to the Hugging Face cache.


In [26]:
"""Fine-tuned T5 translator.

T5 (Raffel et al., 2020, "Exploring the Limits of Transfer Learning with a
Unified Text-to-Text Transformer") is a pretrained encoder-decoder that reads
and writes plain text.
Unlike the LSTM and Transformer, it is not trained from scratch: fit()
fine-tunes the T5_MODEL checkpoint, so it starts out already knowing English.
Its subword tokenizer replaces the word vocabularies, so an unseen word is
spelled out in pieces instead of becoming <unk>.

The input is T5_PREFIX plus the English, or plus the flat or bracketed parse
string when parse_format is set.
Training and decoding come from NeuralTranslator, like the other neural
translators.

Requires: pip install transformers sentencepiece.
The first run downloads the checkpoint (about 300 MB for flan-t5-small) to the
Hugging Face cache.
"""

@dataclass(frozen=True, slots=True, kw_only=True)
class T5Config(TrainingConfig):
    """T5 fine-tuning settings; the defaults come from the settings cell.

    Attributes:
        model_name (str): Hugging Face checkpoint to start from.
        max_source_len (int): Subword tokens kept per input.
        max_target_len (int): Subword tokens kept per gloss in training;
            also caps generated gloss length.

    Example:
        >>> T5Config(epochs=1).model_name
        'google/flan-t5-small'
    """

    model_name: str = T5_MODEL
    max_source_len: int = T5_MAX_SOURCE_LEN
    max_target_len: int = T5_MAX_TARGET_LEN

    # Training defaults for the TrainingConfig fields without one.
    epochs: int = T5_EPOCHS
    batch_size: int = T5_BATCH_SIZE
    learning_rate: float = T5_LEARNING_RATE
    bucket_by_length: bool = T5_BUCKET_BY_LENGTH
    device: str | None = T5_DEVICE


class T5Translator(NeuralTranslator[T5Config, T5ForConditionalGeneration]):
    """Fine-tuned T5 translator: T5Translator(T5Config(), parse_format=...).

    Call fit() before translate().

    Example:
        >>> training = pd.DataFrame({
        ...     "english": ["I eat an orange", "you eat an apple"],
        ...     "gloss": ["X-I EAT ORANGE", "X-YOU EAT APPLE"],
        ... })
        >>> t5 = T5Translator(T5Config(epochs=1)).fit(training, verbose=False)
        >>> len(t5.translate(["I eat an apple"]))
        1
    """

    base_name: ClassVar[str] = "t5"

    def _build_model(self, training: pd.DataFrame) -> T5ForConditionalGeneration:
        """Load the pretrained checkpoint and its tokenizer.

        Args:
            training (pd.DataFrame): Unused: T5's vocabulary is fixed.

        Returns:
            T5ForConditionalGeneration: The pretrained model, ready to
            fine-tune.
        """

        # The tokenizer belongs to the checkpoint; translate() needs it too.
        self.tokenizer: PreTrainedTokenizerBase = AutoTokenizer.from_pretrained(
            self.config.model_name
        )
        return T5ForConditionalGeneration.from_pretrained(self.config.model_name)

    def _sources(self, sentences: list[str]) -> list[str]:
        """The prefixed text T5 reads for each sentence.

        Args:
            sentences (list[str]): English sentences.

        Returns:
            list[str]: T5_PREFIX plus the English or its parse string.
        """

        # Task prefix in T5's usual "translate X to Y:" style.
        T5_PREFIX: str = "translate English to ASL gloss: "

        # With a parse format, T5 reads the serialized parse as plain text.
        texts: list[str] = (
            sentences
            if self.parse_format is None
            else self.parsed_strings(sentences).tolist()
        )
        return [T5_PREFIX + text for text in texts]

    def _targets(self, glosses: list[str]) -> list[str]:
        """The glosses as they are; _batch_loss() tokenizes them."""
        return glosses

    def _tokenize(self, texts: list[str], max_len: int) -> BatchEncoding:
        """Tokenize, truncate and pad a batch of texts onto self.device.

        Args:
            texts (list[str]): Inputs or glosses.
            max_len (int): Subword tokens kept per text.

        Returns:
            BatchEncoding: input_ids and attention_mask, each shape
            (batch, longest_len).
        """
        return self.tokenizer(
            texts,
            max_length=max_len,
            truncation=True,
            padding=True,
            return_tensors="pt",
        ).to(self.device)

    def _batch_loss(
        self, sources: list[str], targets: list[str]
    ) -> tuple[Tensor, int, int]:
        """Mean token loss for one batch, the number of tokens scored, and
        how many of them the model predicted.

        Args:
            sources (list[str]): Prefixed inputs from _sources().
            targets (list[str]): The matching gold glosses.

        Returns:
            tuple[Tensor, int, int]: The scalar loss (still attached to the
            graph), the number of gold subword tokens, and how many of them
            were the model's top prediction.
        """
        # Hugging Face leaves label positions with this id out of the loss.
        IGNORE_INDEX: int = -100

        assert self.model is not None
        labels: Tensor = self._tokenize(targets, self.config.max_target_len)[
            "input_ids"
        ]

        # Padding positions are left out of the loss.
        labels[labels == self.tokenizer.pad_token_id] = IGNORE_INDEX

        # T5 shifts the labels right itself to make the decoder input.
        output = self.model(
            **self._tokenize(sources, self.config.max_source_len), labels=labels
        )
        real: Tensor = labels != IGNORE_INDEX

        # Correct: a real gold token that is also the highest-scoring one.
        right: int = int(((output.logits.argmax(dim=-1) == labels) & real).sum())
        return output.loss, int(real.sum()), right

    def _generate(self, sources: list[str]) -> list[str]:
        """Greedy-decode one batch of prefixed inputs into glosses.

        Args:
            sources (list[str]): Prefixed inputs from _sources().

        Returns:
            list[str]: One space-separated gloss per input, in order.
        """
        assert self.model is not None

        # Greedy decoding (one beam, no sampling), like the other models.
        # (generate() declares a self type that type checkers can't match to
        # T5, though it runs fine.)
        generated = self.model.generate(  # pyright: ignore[reportAttributeAccessIssue]
            **self._tokenize(sources, self.config.max_source_len),
            max_new_tokens=self.config.max_target_len,
            num_beams=1,
            do_sample=False,
        )

        # Keep the gloss tokens as generated: clean-up would glue
        # punctuation to the previous word ("CAR ." -> "CAR.").
        texts: list[str] = self.tokenizer.batch_decode(
            generated, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )
        return [" ".join(text.split()) for text in texts]

# Evaluation

Evaluation setup:

- `data` is split into training, validation and test sets by distinct English sentence, so pairs with the same English always land in the same split and test sentences are never seen in training. Every model uses this same split.
- `EVAL_PARSE_FORMATS` lists the inputs the rule-based translators are scored on: raw English (`None`), the flat parse and the bracketed parse.
- Every sentence is parsed up front, so parsing time isn't charged to whichever model happens to run first.

`eval_df` is the test split, which every translator is scored on.


In [27]:
# The copy and word-substitution cells score once per entry: None reads the
# raw English, "flat" and "bracketed" read that serialized parse tree instead.
EVAL_PARSE_FORMATS: tuple[str | None, ...] = (None, "flat", "bracketed")

# The held-out shares must leave some sentences for training.
if (
    VALIDATION_FRACTION < 0
    or TEST_FRACTION < 0
    or VALIDATION_FRACTION + TEST_FRACTION >= 1
):
    raise ValueError(
        "VALIDATION_FRACTION and TEST_FRACTION must be >= 0 and sum to < 1"
    )

# Split distinct English sentences, not rows, so pairs that share an English
# sentence share a split and test sentences are never trained on.
sentences: pd.Series = pd.Series(data["english"].unique()).sample(
    frac=1.0, random_state=RANDOM_SEED
)
n_test: int = round(len(sentences) * TEST_FRACTION)
n_validation: int = round(len(sentences) * VALIDATION_FRACTION)

# The shuffled sentences are labelled test, then validation, then training.
split_by_sentence: pd.Series = pd.Series(
    ["test"] * n_test
    + ["validation"] * n_validation
    + ["training"] * (len(sentences) - n_test - n_validation),
    index=sentences.to_numpy(),
)
data = data.assign(split=data["english"].map(split_by_sentence))
print(
    data["split"].value_counts().reindex(["training", "validation", "test"]).to_string()
)

# One DataFrame per split; every translator is scored on the test split.
training_df: pd.DataFrame = data[data["split"] == "training"]
validation_df: pd.DataFrame = data[data["split"] == "validation"]
eval_df: pd.DataFrame = data[data["split"] == "test"]

# Every translator uses these same three splits.
# Parse every sentence now, so no model's recorded training or test time
# includes parsing that the cache would spare the models after it.
parse_sentences(data["english"].tolist())

split
training      63327
validation     7916
test           7916


KeyboardInterrupt: 

`_f1()` turns an overlap count into an F1 score, which both ROUGE scores use.
Precision $P$ is the share of the hypothesis's tokens that are in the overlap, and recall $R$ is the share of the reference's tokens that are.
F1 is their harmonic mean, $2PR/(P+R)$, which is high only when both are.
No overlap, or an empty side, scores 0.


In [ ]:
def _f1(overlap: int, hyp_len: int, ref_len: int) -> float:
    """F1 from an overlap count; 0.0 when either side is empty.

    Args:
        overlap (int): Tokens the hypothesis and reference share.
        hyp_len (int): Number of hypothesis tokens.
        ref_len (int): Number of reference tokens.

    Returns:
        float: Harmonic mean of precision and recall, between 0 and 1.

    Example:
        >>> _f1(2, 4, 2)
        0.6666666666666666
    """
    if overlap == 0 or hyp_len == 0 or ref_len == 0:
        return 0.0
    precision: float = overlap / hyp_len
    recall: float = overlap / ref_len
    return 2 * precision * recall / (precision + recall)

## Run evaluation

`run_evaluation()` scores a list of translators on `eval_df`:

1. If `EVAL_LIMIT` is set, it scores a random sample of that many pairs. The sample is seeded with `RANDOM_SEED`, so every translator is scored on the same pairs.
2. Each translator translates all the English in one batched call, and the time it takes is recorded as `test_seconds`.
3. References and predictions are normalized the same way: uppercase and single-spaced, without lone punctuation (`PUNCT_TOKENS`). The `X-`/`DESC-` prefixes are kept, so a wrong prefix counts as an error. Both sides are then scored (see Scoring below).
4. For neural translators, it also records the hyperparameters, the best epoch, and the loss after each epoch.

It writes `metrics.json` (the scores plus the settings of the run), `metrics.tsv`, a `predictions_<system>.tsv` per translator with sentence-level scores, and `training_history.tsv` when a model was trained, all to `out_dir`.
It then prints the metrics table and returns three DataFrames: the metrics, the predictions with their sentence scores, and the training history.

#### Scoring

For each translator, `run_evaluation()` compares the normalized hypotheses (system output) with the references, and computes per-sentence scores and one set of corpus scores, all from 0 to 100.

- **BLEU** measures n-gram precision. For each n from 1 to 4, it counts how many of the hypothesis's n-grams also occur in the reference (each counted at most as often as it occurs there) and divides by the number of hypothesis n-grams. The geometric mean of these four precisions is only high when longer segments match, not just single words. Precision alone would reward very short outputs, so a brevity penalty lowers the score when the hypotheses are shorter in total than the references. The counts are pooled over the whole test set before dividing, so BLEU is a corpus score, not an average of sentence scores. BLEU doesn't account for grammar.
- **chrF** compares character n-grams (1 to 6 characters, spaces ignored) instead of words, and combines their precision and recall in an F-score that weights recall twice as much as precision. Because it matches inside words, it gives partial credit for near misses, such as `SHOE` against `SHOES` or `I` against `X-I`, that BLEU counts as simply wrong. Like BLEU, it is computed over the whole corpus.
- **ROUGE-1** is the F1 of the words the hypothesis and reference share, ignoring order. A word repeated on both sides counts as many times as it appears on the side with fewer copies (`Counter(h) & Counter(r)`).
- **ROUGE-L** is the F1 of the longest common subsequence: the most tokens that appear in both, in the same order but not necessarily next to each other (for `A B C D` and `B D C` it is 2). It rewards correct word order while still allowing gaps. Its length is found by dynamic programming, keeping one row of the table at a time.
- **Exact match** is the share of sentences where the hypothesis and reference are identical.

ROUGE and exact match are computed per sentence and averaged.
The per-sentence scores also go into each system's predictions file, so individual sentences can be inspected.


In [ ]:
def run_evaluation(
    df: pd.DataFrame,
    translators: Sequence[Translator],
    out_dir: Path,
    limit: int | None = EVAL_LIMIT,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Score every translator on df's pairs, save the results, and print them.

    Pass already-constructed (and, for neural models, already-trained)
    translators, so expensive objects are built once.
    Both sides are normalized the same way before scoring.
    Writes metrics.json, metrics.tsv, predictions_<system>.tsv and, if any
    system was trained, training_history.tsv to out_dir.

    Args:
        df (pd.DataFrame): Pairs with "english" and "gloss" columns
            (normally eval_df).
        translators (Sequence[Translator]): Systems to score; names must be
            unique.
        out_dir (Path): Folder for the result files; created if missing.
        limit (int | None): Score a random sample of this many pairs; None
            scores all.
            Defaults to EVAL_LIMIT.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]: The metrics, one row per
        system (indexed by name) with its parse format, corpus scores,
        training_seconds (0 for rule-based systems) and test_seconds (time
        to translate the scored pairs).
        The predictions, one row per (system, pair), keeping the original
        row ids, with sentence-level scores.
        The per-epoch losses and token accuracies of trained systems
        (empty if none were trained).

    Raises:
        ValueError: If df is empty or two translators share a name.

    Example:
        >>> metrics, predictions, training = run_evaluation(
        ...     eval_df, [CopyTranslator()], out_dir=RESULTS_DIR / "demo"
        ... )  # prints the metrics table
        >>> metrics.loc["copy", "bleu"]  # a float from 0 to 100
    """

    # Lone tokens dropped from both sides before scoring.
    PUNCT_TOKENS: frozenset[str] = frozenset({".", ",", "?", "!", ";", ":"})

    if df.empty:
        raise ValueError("No pairs to evaluate")
    names: list[str] = [translator.name for translator in translators]
    if len(set(names)) != len(names):
        raise ValueError(f"Translator names must be unique: {names}")

    # Rows are sorted alphabetically, so sample instead of taking head().
    if limit is not None and limit < len(df):
        df = df.sample(n=limit, random_state=RANDOM_SEED).sort_index()

    metric_rows: list[dict[str, object]] = []
    prediction_frames: list[pd.DataFrame] = []
    training_frames: list[pd.DataFrame] = []

    # Hyperparameters and best epoch of each neural system, saved with the
    # metrics.
    model_configs: dict[str, dict[str, Any]] = {}
    for translator in translators:
        # One batched call per system, so parsers and models run in batches.
        predictions: pd.DataFrame = df[["english", "gloss"]].rename(
            columns={"gloss": "reference"}
        )

        # Time only the translation; scoring is the same work for every system.
        start: float = time.perf_counter()
        predictions["prediction"] = translator.translate(df["english"].tolist())
        test_seconds: float = time.perf_counter() - start

        # Normalize both sides identically so no system is favored: uppercase,
        # collapse whitespace, and drop lone punctuation.
        # The X-/DESC- prefixes are kept, so a wrong prefix counts as an error.
        # str() tells type checkers each value (possibly NA) is a str.
        for column, source in (("target", "reference"), ("hypothesis", "prediction")):
            predictions[column] = [
                " ".join(
                    token
                    for token in str(text).upper().split()
                    if token not in PUNCT_TOKENS
                )
                for text in predictions[source]
            ]

        # Sentence scores.
        # ROUGE-1 counts shared words (Counter & keeps the smaller count).
        # ROUGE-L counts the longest common subsequence: shared words in the
        # same order, gaps allowed.
        rows: list[dict[str, float]] = []
        for hyp, ref in zip(
            predictions["hypothesis"], predictions["target"], strict=True
        ):
            h, r = hyp.split(), ref.split()

            # Longest common subsequence by dynamic programming: previous[j]
            # is its length for the hypothesis so far and the first j
            # reference tokens.
            # Only the previous row is kept.
            previous: list[int] = [0] * (len(r) + 1)
            for token_h in h:
                current: list[int] = [0] * (len(r) + 1)
                for j, token_r in enumerate(r, start=1):
                    # A match extends the diagonal; otherwise carry the best.
                    current[j] = (
                        previous[j - 1] + 1
                        if token_h == token_r
                        else max(previous[j], current[j - 1])
                    )
                previous = current
            rows.append(
                {
                    "rouge_1": 100
                    * _f1(sum((Counter(h) & Counter(r)).values()), len(h), len(r)),
                    "rouge_l": 100 * _f1(previous[-1], len(h), len(r)),
                    "exact_match": 100.0 * (hyp == ref),
                }
            )
        sentences: pd.DataFrame = pd.DataFrame(rows, index=predictions.index)

        # Corpus scores: BLEU and chrF from sacreBLEU, sentence averages for
        # the rest, rounded to 2 decimals.
        # tokenize="none": gloss is already space-tokenized, so sacreBLEU's
        # default tokenizer would only risk splitting tokens like "X-I".
        hyp_list: list[str] = predictions["hypothesis"].tolist()
        ref_lists: list[list[str]] = [predictions["target"].tolist()]
        corpus: dict[str, float] = {
            "bleu": sacrebleu.corpus_bleu(hyp_list, ref_lists, tokenize="none").score,
            "chrf": sacrebleu.corpus_chrf(hyp_list, ref_lists).score,
            **{name: float(sentences[name].mean()) for name in sentences.columns},
        }

        # Sentence scores join the predictions; corpus scores make one row.
        prediction_frames.append(
            predictions.join(sentences).assign(system=translator.name)
        )
        metric_rows.append(
            {
                "system": translator.name,
                "parse_format": translator.parse_format or "none",
                **{name: round(value, 2) for name, value in corpus.items()},
                "training_seconds": translator.training_seconds,
                "test_seconds": test_seconds,
            }
        )

        # Neural translators also have a loss history and hyperparameters.
        if isinstance(translator, NeuralTranslator):
            model_configs[translator.name] = {
                **asdict(translator.config),
                "best_epoch": translator.best_epoch,
            }
            if not translator.history.empty:
                training_frames.append(
                    translator.history.assign(system=translator.name)
                )

    metrics: pd.DataFrame = pd.DataFrame(metric_rows).set_index("system")
    all_predictions: pd.DataFrame = pd.concat(prediction_frames)
    training: pd.DataFrame = (
        pd.concat(training_frames, ignore_index=True)
        if training_frames
        else pd.DataFrame()
    )

    # Save metrics (with the run's settings) as JSON and TSV.
    out_dir.mkdir(parents=True, exist_ok=True)
    config: dict[str, object] = {
        "num_pairs": len(df),
        "limit": limit,
        "models": model_configs,
    }
    (out_dir / "metrics.json").write_text(
        json.dumps(
            {"config": config, "systems": metrics.to_dict(orient="index")},
            indent=2,
        ),
        encoding="utf-8",
    )
    metrics.to_csv(out_dir / "metrics.tsv", sep="\t")

    # One predictions file per system; QUOTE_NONE keeps the text unquoted.
    for name, predictions in all_predictions.groupby("system", sort=False):
        predictions.drop(columns="system").to_csv(
            out_dir / f"predictions_{name}.tsv",
            sep="\t",
            index_label="row",
            quoting=csv.QUOTE_NONE,
            escapechar="\\",
            lineterminator="\n",
            float_format="%.2f",
        )

    # Per-epoch losses and accuracies, when any system was trained.
    if not training.empty:
        training.to_csv(
            out_dir / "training_history.tsv", sep="\t", index=False, float_format="%.4f"
        )

    print(metrics.to_string(float_format=lambda value: f"{value:.2f}"))
    print(f"\nWrote results to {out_dir}")
    return metrics, all_predictions, training

`evaluations` and `trained_models` collect results, keyed by system name, as the translator cells below run; re-running this cell empties both.

`train_and_evaluate()` is what each neural cell calls.
It trains the translator on `training_df`, with early stopping on `validation_df`, stores the trained model, and scores it with `run_evaluation()` into `RESULTS_DIR/<name>`.

Each trained model is also saved to `MODEL_CACHE_DIR/<name>.pt`.
When the cell runs again, the saved model is loaded instead of trained, as long as its settings and the training and validation data are unchanged; it is still scored, so its results are written again.
The training time in its results is from when it was trained.
Changes to a model's code are not detected: after editing a model class, set `RETRAIN_MODELS = True` or delete `MODEL_CACHE_DIR`.


In [ ]:
# Every translator cell below adds its result here, and each neural model
# cell adds its trained model; re-running this cell clears both.
evaluations: dict[str, tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]] = {}
trained_models: dict[str, NeuralTranslator[Any, Any]] = {}


def train_and_evaluate(translator: NeuralTranslator[Any, Any]) -> None:
    """Train one neural translator, or load it, then score it and record both.

    A trained translator is saved to MODEL_CACHE_DIR/<name>.pt.
    Later calls load it instead of training again, as long as its config and
    the training and validation rows are unchanged and RETRAIN_MODELS is False.
    It is scored on eval_df with run_evaluation() into RESULTS_DIR/<name>, and
    the result and the model are stored in evaluations and trained_models under
    its name.

    Args:
        translator (NeuralTranslator[Any, Any]): An untrained translator,
            e.g. LSTMTranslator(LSTMConfig(), parse_format="flat").

    Example:
        >>> train_and_evaluate(LSTMTranslator(LSTMConfig(epochs=1)))
        ... # prints the losses and the metrics table
        >>> trained_models["lstm"].best_epoch
        1
    """
    path: Path = MODEL_CACHE_DIR / f"{translator.name}.pt"

    # Fingerprint the config and the exact training and validation rows; any
    # difference means a saved model is stale.
    # Model code is not part of it: after changing a model class, set
    # RETRAIN_MODELS or delete MODEL_CACHE_DIR.
    digest = hashlib.sha256(repr(translator.config).encode("utf-8"))
    for frame in (training_df, validation_df):
        digest.update(frame.to_csv().encode("utf-8"))
    fingerprint: str = digest.hexdigest()

    # The saved file, if there is one and it may be used.
    # It holds the whole translator (vocabularies and tokenizer included), not
    # just tensors, so torch needs weights_only=False to load it.
    saved: dict[str, Any] | None = None
    if path.is_file() and not RETRAIN_MODELS:
        saved = torch.load(path, weights_only=False)

    # Reuse the saved translator if it matches; otherwise train and save.
    if saved is not None and saved["fingerprint"] == fingerprint:
        translator = saved["translator"]
        print(f"Loaded {translator.name} from {path}")
    else:
        translator.fit(training_df, validation_df)
        path.parent.mkdir(parents=True, exist_ok=True)
        torch.save({"fingerprint": fingerprint, "translator": translator}, path)
        print(f"Saved {translator.name} to {path}")

    trained_models[translator.name] = translator
    evaluations[translator.name] = run_evaluation(
        eval_df, [translator], out_dir=RESULTS_DIR / translator.name
    )

# Run translators


Scores the copy translator once per parse format (raw English, flat, bracketed).


In [ ]:
# Copy translator, once per parse format (raw English, flat, bracketed).
# No model or spaCy pipeline is needed unless a parse format is used.
evaluations["copy"] = run_evaluation(
    eval_df,
    [CopyTranslator(parse_format=fmt) for fmt in EVAL_PARSE_FORMATS],
    out_dir=RESULTS_DIR / "copy",
)

Scores the word-substitution baseline once per parse format.


In [ ]:
# Word-substitution baseline, once per parse format.
# The plain version loads a lemmatizer; the parse versions read
# parse_sentences()'s parses.
evaluations["word_substitution"] = run_evaluation(
    eval_df,
    [WordSubstitutionTranslator(parse_format=fmt) for fmt in EVAL_PARSE_FORMATS],
    out_dir=RESULTS_DIR / "word_substitution",
)

LSTM on the raw English.
Like every neural cell below, it trains on `training_df`, stops early once the validation loss stops improving, and scores `eval_df`.
The neural cells are the slow ones on the full data.


In [ ]:
# LSTM reading the raw English.
# Each neural cell trains on training_df, stops early once the validation_df
# loss stops improving, and scores eval_df.
# Neural cells are the slow ones on the full data.
train_and_evaluate(LSTMTranslator(LSTMConfig()))

LSTM on the flat parse (lemma, POS, dependency label and head per token).


In [ ]:
# LSTM reading the flat parse (lemma, POS, dep, head per token).
train_and_evaluate(LSTMTranslator(LSTMConfig(), parse_format="flat"))

LSTM on the bracketed parse tree.


In [ ]:
# LSTM reading the bracketed parse tree.
train_and_evaluate(LSTMTranslator(LSTMConfig(), parse_format="bracketed"))

Transformer on the raw English, trained and scored like the LSTM.


In [ ]:
# Transformer reading the raw English (trained and scored like the LSTM).
train_and_evaluate(TransformerTranslator(TransformerConfig()))

Transformer on the flat parse.


In [ ]:
# Transformer reading the flat parse (lemma, POS, dep, head per token).
train_and_evaluate(TransformerTranslator(TransformerConfig(), parse_format="flat"))

Transformer on the bracketed parse tree.


In [ ]:
# Transformer reading the bracketed parse tree.
train_and_evaluate(TransformerTranslator(TransformerConfig(), parse_format="bracketed"))

T5 on the raw English, fine-tuned on the GPU when there is one (`T5_DEVICE`).
The T5 cells are the slowest on the full data.


In [ ]:
# T5 reading the raw English: fine-tunes the pretrained T5_MODEL on the GPU when
# there is one (T5_DEVICE).
# The slowest cells on the full data.
train_and_evaluate(T5Translator(T5Config()))

T5 on the flat parse.


In [ ]:
# T5 reading the flat parse (lemma, POS, dep, head per token).
train_and_evaluate(T5Translator(T5Config(), parse_format="flat"))

T5 on the bracketed parse tree.


In [ ]:
# T5 reading the bracketed parse tree.
train_and_evaluate(T5Translator(T5Config(), parse_format="bracketed"))

Side-by-side outputs from every trained model, first on hand-written sentences, then on held-out test pairs with their reference glosses.
Run the LSTM, Transformer or T5 cells first.
This is for getting a feel for how the models differ; the metrics are the real comparison.


In [ ]:
# Side-by-side outputs from every trained model (run the LSTM, Transformer
# and/or T5 cells first).
# Use it to get a feel for how the models differ; the metrics above are the real
# comparison.

# Hand-written sentences in the dataset's style (lowercase except "I",
# punctuation split off), each aimed at one way gloss differs from English.
MODEL_TEST_SAMPLES: pd.DataFrame = pd.DataFrame(
    [
        ("we support the proposal .", "pronoun gets X-, article dropped"),
        ("the commission is not ready .", "negation gets DESC-, be kept or dropped"),
        ("this is a very important report .", "adverb and adjective get DESC-"),
        (
            "they have voted against their own interests .",
            "pronouns, possessive, tense",
        ),
        (
            "I thank the rapporteur for her excellent work .",
            "common parliamentary phrase",
        ),
        ("what will the council do about it ?", "question"),
        ("the zebra danced quietly in the moonlight .", "words unseen in training"),
        (
            "mr president , ladies and gentlemen , the situation in the region is "
            "serious and we must act quickly before it becomes worse .",
            "long sentence",
        ),
    ],
    columns=["english", "tests"],
)

# A few real held-out pairs, so there is a reference gloss to compare against.
MODEL_TEST_REFERENCES: pd.DataFrame = (
    eval_df.sample(n=min(4, len(eval_df)), random_state=RANDOM_SEED)[
        ["english", "gloss"]
    ]
    .rename(columns={"gloss": "reference"})
    .reset_index(drop=True)
)

# Whichever model cells have been run, in the order they ran.
models: list[Translator] = list(trained_models.values())

# Nothing to compare without at least one trained model.
if not models:
    raise RuntimeError("Run at least one LSTM, Transformer or T5 cell first")

# Print every model's output under each sentence: first the hand-written
# probes (with what each tests), then the held-out pairs (with reference).
print("=" * 30, "hand-written sentences", "=" * 30)
model_test_df: pd.DataFrame = translator_test(models, MODEL_TEST_SAMPLES)
print("=" * 30, "held-out test pairs", "=" * 30)
model_reference_df: pd.DataFrame = translator_test(models, MODEL_TEST_REFERENCES)

One metrics table for every translator cell run so far, also saved as `RESULTS_DIR/metrics.tsv`.
Each cell's own predictions and `metrics.json` stay in its subfolder.


In [ ]:
# One metrics table for every translator cell run so far, also saved as
# RESULTS_DIR/metrics.tsv.
# Each cell's predictions and metrics.json stay in its own subfolder.
if not evaluations:
    raise RuntimeError("Run at least one translator cell first")
combined_metrics: pd.DataFrame = pd.concat(
    [metrics for metrics, _, _ in evaluations.values()]
)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
combined_metrics.to_csv(RESULTS_DIR / "metrics.tsv", sep="\t")
print(combined_metrics.to_string(float_format=lambda value: f"{value:.2f}"))
print(
    f"\nWrote combined metrics for {', '.join(evaluations)} to {RESULTS_DIR / 'metrics.tsv'}"
)

Loss and token accuracy per epoch for every LSTM and Transformer system (T5 trains for only three epochs), read from each system's `training_history.tsv` and saved as `RESULTS_DIR/training_curves.png`: one row each for the LSTM's loss, the LSTM's accuracy, the Transformer's loss and the Transformer's accuracy, and one column per input (raw, flat, bracketed).

Token accuracy is the share of gold gloss tokens the model ranks first, with teacher forcing (each position sees the gold tokens before it), so it is higher than the share of whole glosses translated correctly.
A model loaded from `MODEL_CACHE_DIR` keeps the history of the run that trained it, so a model trained before accuracy was recorded shows no accuracy curve until it is retrained (`RETRAIN_MODELS = True`).


In [ ]:
# One row per model and measure (LSTM loss, LSTM accuracy, Transformer loss,
# Transformer accuracy) and one column per input (raw, flat, bracketed), so
# each column compares the two architectures on the same input.
# T5 is left out: its three epochs make too short a curve to show much.
CURVE_MODELS: tuple[str, ...] = ("lstm", "transformer")
# Each measure with its axis label and legend position: loss falls and
# accuracy rises, so these corners stay clear of the curves.
CURVE_METRICS: tuple[tuple[str, str, str], ...] = (
    ("loss", "cross-entropy", "upper right"),
    ("accuracy", "token accuracy", "lower right"),
)

# Training in blue and validation in orange, a pair that stays distinct with
# color blindness.
CURVE_COLORS: dict[str, str] = {"training": "#2a78d6", "validation": "#eb6834"}

fig, axes = plt.subplots(
    len(CURVE_MODELS) * len(CURVE_METRICS),
    len(EVAL_PARSE_FORMATS),
    figsize=(10, 11),
)
for model_index, model in enumerate(CURVE_MODELS):
    for column, parse_format in enumerate(EVAL_PARSE_FORMATS):
        system: str = f"{model}+{parse_format}" if parse_format else model
        history_path: Path = RESULTS_DIR / system / "training_history.tsv"
        history: pd.DataFrame = (
            pd.read_csv(history_path, sep="\t")
            if history_path.is_file()
            else pd.DataFrame()
        )
        for metric_index, (metric, y_label, legend_corner) in enumerate(
            CURVE_METRICS
        ):
            ax = axes[model_index * len(CURVE_METRICS) + metric_index, column]
            ax.set(title=f"{system}: {metric}", xlabel="epoch")

            # Whole epochs only, so short runs don't get half-ticks.
            ax.xaxis.set_major_locator(MaxNLocator(integer=True))

            # The axis label only on the first column; each row shares a
            # measure.
            if column == 0:
                ax.set_ylabel(y_label)

            # A note instead of curves for an untrained system, or for a
            # history from before accuracy was recorded.
            if f"training_{metric}" not in history:
                ax.text(
                    0.5,
                    0.5,
                    "not trained" if history.empty else "not recorded:\nretrain to add",
                    ha="center",
                    va="center",
                    color="gray",
                    transform=ax.transAxes,
                )
                continue

            # One line per split, with a marker on each epoch.
            for split, color in CURVE_COLORS.items():
                ax.plot(
                    history["epoch"],
                    history[f"{split}_{metric}"],
                    "o-",
                    color=color,
                    markersize=3,
                    label=split,
                )
            ax.legend(loc=legend_corner, fontsize="small")

plt.tight_layout()

# A tight bounding box keeps the titles at the edges from being cut off.
fig.savefig(RESULTS_DIR / "training_curves.png", dpi=200, bbox_inches="tight")
plt.show()


### Translate your own sentence

Translates `user_sentence` with every translator.
Write the sentence in the dataset's style: lowercase except "I", with punctuation split off by spaces.


In [ ]:
# The sentence to translate, in the dataset's style.
user_sentence: str = "computational linguistics is a fun course ."

# The rule-based translators in every parse format, then the trained models.
all_translators: list[Translator] = [
    translator_class(parse_format=parse_format)
    for translator_class in (CopyTranslator, WordSubstitutionTranslator)
    for parse_format in EVAL_PARSE_FORMATS
]
all_translators += trained_models.values()

# One line per translator, with the names lined up.
print(user_sentence)
for translator in all_translators:
    print(f"  {translator.name:<28} -> {translator.translate([user_sentence])[0]}")

# A reminder when no neural model has been trained yet.
if not trained_models:
    print("\n(No neural models yet: run the LSTM, Transformer or T5 cells.)")